In [1]:
# ============================================================
# STEP 1: Download and load the Ott et al. (2011) dataset
# ============================================================
# This dataset has 1,600 hotel reviews:
#   - 800 truthful (real) reviews
#   - 800 deceptive (fake) reviews
# It is the gold-standard corpus cited in the literature review.

import pandas as pd   # pandas organises data into tables (like Excel)

# The dataset is hosted publicly on GitHub in CSV format.
url = "https://raw.githubusercontent.com/rajsiddarth/Text_Analytics/master/deceptive-opinion.csv"

# Read the CSV file into a pandas "DataFrame" (a table we can work with)
df = pd.read_csv(url)

# Show the size of the dataset: (rows, columns)
print("Dataset shape:", df.shape)

# Show the first 5 rows so we can see what the data looks like
df.head()


Dataset shape: (1600, 5)


,deceptive,hotel,polarity,source,text
0,truthful,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...
1,truthful,hyatt,positive,TripAdvisor,Triple A rate with upgrade to view room was le...
2,truthful,hyatt,positive,TripAdvisor,This comes a little late as I'm finally catchi...
3,truthful,omni,positive,TripAdvisor,The Omni Chicago really delivers on all fronts...
4,truthful,hyatt,positive,TripAdvisor,I asked for a high floor away from the elevato...


In [2]:
# ============================================================
# STEP 3: Preprocess the text and prepare labels
# ============================================================

# 3a. Turn the labels into numbers.
# Machine learning models work with numbers, not words.
# We map:  deceptive -> 1  (the thing we want to detect)
#          truthful  -> 0
df['label'] = df['deceptive'].map({'deceptive': 1, 'truthful': 0})

# Quick check that the mapping worked
print("Label counts (1 = deceptive, 0 = truthful):")
print(df['label'].value_counts())
print()

# 3b. Basic text cleaning function.
# We lowercase everything and remove extra spaces so that
# "Great" and "great" are treated as the same word.
import re  # 're' is Python's tool for text pattern matching

def clean_text(text):
    text = text.lower()                      # make everything lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)    # keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces
    return text

# Apply the cleaning to every review, storing the result in a new column
df['clean_text'] = df['text'].apply(clean_text)

# 3c. Show the difference: original vs cleaned, for the first review
print("--- ORIGINAL ---")
print(df['text'].iloc[0][:200])   # first 200 characters
print()
print("--- CLEANED ---")
print(df['clean_text'].iloc[0][:200])


Label counts (1 = deceptive, 0 = truthful):
label
0    800
1    800
Name: count, dtype: int64

--- ORIGINAL ---
We stayed for a one night getaway with family on a thursday. Triple AAA rate of 173 was a steal. 7th floor room complete with 44in plasma TV bose stereo, voss and evian water, and gorgeous bathroom(no

--- CLEANED ---
we stayed for a one night getaway with family on a thursday triple aaa rate of was a steal th floor room complete with in plasma tv bose stereo voss and evian water and gorgeous bathroom no tub but wa


In [3]:
# ============================================================
# STEP 5: Train and evaluate the three classifiers
# ============================================================
# Your proposal compares three models:
#   - Logistic Regression (a linear baseline)
#   - Random Forest        (an ensemble of decision trees)
#   - Naive Bayes          (a probabilistic model, classic for text)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Put the three models in a dictionary so we can loop through them
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Naive Bayes": MultinomialNB()
}

# We'll collect each model's scores here so we can compare them at the end
results = []

# Train each model, then test it on the unseen test set
for name, model in models.items():
    model.fit(X_train, y_train)          # LEARN from the training data
    y_pred = model.predict(X_test)       # PREDICT on the unseen test data

    # Measure performance using your 4 proposal metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": round(acc, 3),
        "Precision": round(prec, 3),
        "Recall": round(rec, 3),
        "F1-score": round(f1, 3)
    })

# Show the results as a neat comparison table
results_df = pd.DataFrame(results)
print("Model comparison on the test set:")
results_df


NameError: name 'X_train' is not defined

In [ ]:
# ============================================================
# STEP 6: Which words most signal DECEPTIVE vs TRUTHFUL reviews?
# ============================================================
# Logistic Regression assigns a weight to every word/feature.
#   Large POSITIVE weight -> pushes toward "deceptive" (label 1)
#   Large NEGATIVE weight -> pushes toward "truthful"  (label 0)
# So the strongest indicators are the words with the most extreme weights.

import numpy as np

# Re-train Logistic Regression on its own so we can inspect it
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Get the list of words/features that TF-IDF created
feature_names = np.array(vectorizer.get_feature_names_out())

# Get the learned weight (coefficient) for each word
coefficients = lr_model.coef_[0]

# Sort the weights: highest = most "deceptive", lowest = most "truthful"
top_n = 15  # how many words to show for each class

# The top deceptive indicators (largest positive weights)
top_deceptive_idx = np.argsort(coefficients)[-top_n:][::-1]
# The top truthful indicators (largest negative weights)
top_truthful_idx = np.argsort(coefficients)[:top_n]

print("=" * 45)
print("TOP", top_n, "WORDS SIGNALLING A *DECEPTIVE* REVIEW")
print("=" * 45)
for i in top_deceptive_idx:
    print(f"  {feature_names[i]:20s}  (weight: {coefficients[i]:.3f})")

print()
print("=" * 45)
print("TOP", top_n, "WORDS SIGNALLING A *TRUTHFUL* REVIEW")
print("=" * 45)
for i in top_truthful_idx:
    print(f"  {feature_names[i]:20s}  (weight: {coefficients[i]:.3f})")

TOP 15 WORDS SIGNALLING A *DECEPTIVE* REVIEW
  chicago               (weight: 3.100)
  my                    (weight: 2.684)
  luxury                (weight: 1.615)
  hotel                 (weight: 1.342)
  hilton                (weight: 1.223)
  when                  (weight: 1.087)
  was                   (weight: 1.073)
  my room               (weight: 1.052)
  chicago hotel         (weight: 1.050)
  experience            (weight: 1.042)
  my stay               (weight: 1.006)
  vacation              (weight: 0.964)
  luxurious             (weight: 0.962)
  be                    (weight: 0.921)
  staying               (weight: 0.889)

TOP 15 WORDS SIGNALLING A *TRUTHFUL* REVIEW
  location              (weight: -1.934)
  floor                 (weight: -1.481)
  on                    (weight: -1.412)
  great                 (weight: -1.329)
  is                    (weight: -1.299)
  small                 (weight: -1.240)
  no                    (weight: -1.235)
  we                   

In [ ]:
# ============================================================
# STEP 7: Save the trained model and vectorizer for reuse
# ============================================================
# We save TWO things as a matched pair:
#   1. the best classifier (so we don't retrain every time)
#   2. the TF-IDF vectorizer (so new text is converted the SAME way)
# Both are needed together to classify any new review later.

import joblib   # joblib saves Python objects (models) to files efficiently

# Re-train Logistic Regression on the full training set to be sure
# we're saving a clean, fully-trained model.
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train, y_train)

# Save the model to a file
joblib.dump(final_model, "fake_review_model.pkl")

# Save the vectorizer to a file
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Saved successfully:")
print("  - fake_review_model.pkl   (the trained classifier)")
print("  - tfidf_vectorizer.pkl    (the vocabulary/feature maker)")

Saved successfully:
  - fake_review_model.pkl   (the trained classifier)
  - tfidf_vectorizer.pkl    (the vocabulary/feature maker)


In [1]:
# ============================================================
# CHECK: Can we load the Namibian CSV correctly?
# ============================================================
import pandas as pd

# Load your CSV
nam = pd.read_csv("namibian_reviews.csv")

# Show how many rows and what the columns are
print("Number of rows:", len(nam))
print("Columns:", nam.columns.tolist())
print()

# Show everything currently in the file
nam

FileNotFoundError: [Errno 2] No such file or directory: 'namibian_reviews.csv'

In [2]:
# Show which folder the notebook is running from, and what files are here
import os
print("Notebook is running in this folder:")
print(os.getcwd())
print()
print("Files I can see here:")
for f in os.listdir():
    print("  ", f)

Notebook is running in this folder:
c:\Users\Acer\Documents\school\Research Project\Fake Review Detector

Files I can see here:
   01_explore_data.ipynb
   fake_review_model.pkl
   namibian_reviews_template_1.csv
   tfidf_vectorizer.pkl
   venv


In [7]:
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")
print("Number of rows:", len(nam))
print("Columns:", nam.columns.tolist())
nam

Number of rows: 30
Columns: ['record_id', 'source', 'product_category', 'text', 'label_annotator1', 'label_annotator2', 'final_label']


,record_id,source,product_category,text,label_annotator1,label_annotator2,final_label
0,1,google_maps,beauty,"Great environment, good music and a lovely bar...",NaN,NaN,NaN
1,2,google_maps,beauty,I never experienced this from any other salon ...,NaN,NaN,NaN
2,3,google_maps,beauty,I had a very disappointing experience at this ...,NaN,NaN,NaN
3,4,google_maps,beauty,I was also not provided with proof of payment....,NaN,NaN,NaN
4,5,google_maps,beauty,I hope this review helps others make an inform...,NaN,NaN,NaN
5,6,google_maps,beauty,"The salon looks fancy at first glance, but the...",NaN,NaN,NaN
6,7,google_maps,beauty,I had a great experience at this salon. The st...,NaN,NaN,NaN
7,8,google_maps,beauty,I had a terrible experience at this salon. The...,NaN,NaN,NaN
8,9,google_maps,beauty,I was very impressed with the level of service...,NaN,NaN,NaN
9,10,google_maps,mobile_phones,They were friendly and stuff but I honestly wo...,NaN,NaN,NaN


In [8]:

import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")

print("Total rows:", len(nam))
print()

# Check for any leftover placeholder rows
placeholders = nam[nam['text'].str.contains("replace this", case=False, na=False)]
print("Leftover placeholder rows:", len(placeholders))
print()

# Check the spread across sources and categories
print("By source:")
print(nam['source'].value_counts())
print()
print("By category:")
print(nam['product_category'].value_counts())
print()

# Check for any very short reviews (under 3 words) that will get dropped later
nam['word_count'] = nam['text'].str.split().str.len()
short = nam[nam['word_count'] < 3]
print("Reviews under 3 words (will be dropped):", len(short))

Total rows: 30

Leftover placeholder rows: 0

By source:
source
google_maps    30
Name: count, dtype: int64

By category:
product_category
mobile_phones         10
beauty                 9
car_dealerships        8
smart_shop_Namibia     3
Name: count, dtype: int64

Reviews under 3 words (will be dropped): 1


In [1]:
# ============================================================
# PASTE HELPER: turn raw reviews into clean CSV rows
# ============================================================
# HOW TO USE:
#  1. Set the category and source below
#  2. Paste your reviews between the triple quotes - ONE REVIEW PER LINE
#  3. Run the cell - it appends them to your CSV automatically,
#     handling all the commas, quotes and numbering for you.

import pandas as pd
import os

# ---- SETTINGS: change these for each batch ----
category = "salon_beauty"     # what kind of business this batch is
source   = "google_maps"      # where you got them
# ------------------------------------------------

# Paste reviews here, ONE PER LINE, between the triple quotes.
# Don't worry about commas or quotes - just paste the raw text.
raw_reviews = """
paste your first review here
paste your second review here
paste your third review here
"""

# --- the code does the rest ---
csv_file = "namibian_reviews.csv"

# Load existing data (or start fresh if file is empty/new)
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
else:
    existing = pd.DataFrame(columns=["record_id","source","product_category","text",
                                     "label_annotator1","label_annotator2","final_label"])

# Split the pasted block into individual reviews (one per line)
new_lines = [line.strip() for line in raw_reviews.strip().split("\n") if line.strip()]

# Figure out the next record_id number
next_id = (existing["record_id"].max() + 1) if len(existing) > 0 else 1

# Build the new rows
new_rows = []
for i, review in enumerate(new_lines):
    new_rows.append({
        "record_id": int(next_id + i),
        "source": source,
        "product_category": category,
        "text": review,
        "label_annotator1": "",
        "label_annotator2": "",
        "final_label": ""
    })

# Add them and save
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(csv_file, index=False)

print(f"Added {len(new_rows)} reviews as category '{category}'.")
print(f"Total reviews now: {len(updated)}")

Added 3 reviews as category 'salon_beauty'.
Total reviews now: 34


In [1]:

 # ============================================================
# PASTE HELPER: turn raw reviews into clean CSV rows
# ============================================================
# HOW TO USE:
#  1. Set the category and source below
#  2. Paste your reviews between the triple quotes - ONE REVIEW PER LINE
#  3. Run the cell - it appends them to your CSV automatically,
#     handling all the commas, quotes and numbering for you.

import pandas as pd
import os

# ---- SETTINGS: change these for each batch ----
category = "salon_beauty"     # what kind of business this batch is
source   = "google_maps"      # where you got them
# ------------------------------------------------

# Paste reviews here, ONE PER LINE, between the triple quotes.
# Don't worry about commas or quotes - just paste the raw text.
raw_reviews = """
100% PERFECT SALON!!! BEST HAIR!!! BEST NAILS!!! BEST SERVICE!!! FIVE STARS!!!
Ladies, STOP LOOKING!!! This is the ONLY salon you need!!! BEST IN NAMIBIA!!! ❤️❤️❤️
OMG!!! AMAZING!!! AMAZING!!! AMAZING!!! My hair is PERFECT and the staff are angels!!!
I didn't even have an appointment but they gave me the BEST hairstyle ever!!! 100000% recommended!!!
THE BEST BEAUTY SALON ON EARTH!!! Everyone must come here!!! You will never regret it!!!
Best service in Namibia guaranteed!!! My nails are PERFECT!!!
CHEAPEST AND BEST SALON!!! Amazing workers!!! Amazing prices!!! Amazing hair!!!
WOW!!! I LOVE THIS PLACE!!! They made me look like a celebrity!!! BEST SALON EVER!!!
Book now!!! Don't even think twice!!! This is the BEST salon in the whole country!!!
I received exactly the hairstyle I requested. The appointment took about three hours.
The staff member was patient and helped me choose a nail colour.
The service was okay. Nothing special but I was happy with my hairstyle.
Clean place and friendly staff. I would recommend it for a simple haircut.
The haircut was good and exactly what I showed the barber.
I liked the service and the staff were welcoming. The price was also what we agreed before the appointment.
The salon was busy but the workers remained friendly. My nails came out nicely.
The braids were neat but I had to wait longer than expected. Overall I'm satisfied.
I booked through WhatsApp and received confirmation the same day. The service was good.
Good experience. The stylist suggested a different style because of my hair type and I liked the result.
Staff were polite and the salon was clean. My hair treatment took around two hours.
The makeup artist listened to what I wanted and the makeup lasted through the event.
Bay
I liked the hairstyle and the price was reasonable. Will probably come again.
The stylist understood what I wanted and the haircut came out nicely.
Girls, don't waste your money somewhere else. This is where you need to go. Service is 10/10 and the prices are good.
I literally walked out looking like a different person. These ladies have magic hands, no jokes.
If you are still doing your hair somewhere else then you are missing out. These ladies are the real deal.
Hai, my girl was not lying when she recommended this salon. Hair was giving, nails were giving, everything was giving.
Honestly, these ladies know their story. I showed one picture and they made it even better.
Yoh ladies, this place is the plug. My hair came out lekker and everyone at work was asking where I did it.

"""

# --- the code does the rest ---
csv_file = "namibian_reviews.csv"

# Load existing data (or start fresh if file is empty/new)
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
else:
    existing = pd.DataFrame(columns=["record_id","source","product_category","text",
                                     "label_annotator1","label_annotator2","final_label"])

# Split the pasted block into individual reviews (one per line)
new_lines = [line.strip() for line in raw_reviews.strip().split("\n") if line.strip()]

# Figure out the next record_id number
next_id = (existing["record_id"].max() + 1) if len(existing) > 0 else 1

# Build the new rows
new_rows = []
for i, review in enumerate(new_lines):
    new_rows.append({
        "record_id": int(next_id + i),
        "source": source,
        "product_category": category,
        "text": review,
        "label_annotator1": "",
        "label_annotator2": "",
        "final_label": ""
    })

# Add them and save
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(csv_file, index=False)

print(f"Added {len(new_rows)} reviews as category '{category}'.")
print(f"Total reviews now: {len(updated)}")

Added 30 reviews as category 'salon_beauty'.
Total reviews now: 60


In [2]:
# ============================================================
# PASTE HELPER: turn raw reviews into clean CSV rows
# ============================================================
# HOW TO USE:
#  1. Set the category and source below
#  2. Paste your reviews between the triple quotes - ONE REVIEW PER LINE
#  3. Run the cell - it appends them to your CSV automatically,
#     handling all the commas, quotes and numbering for you.

import pandas as pd
import os

# ---- SETTINGS: change these for each batch ----
category = "cell_shops"     # what kind of business this batch is
source   = "google_maps"      # where you got them
# ------------------------------------------------

# Paste reviews here, ONE PER LINE, between the triple quotes.
# Don't worry about commas or quotes - just paste the raw text.
raw_reviews = """
The staff were friendly and helped me choose a phone within my budget. I did not wait very long.
I bought a charger here last week. The service was okay and the lady at the counter was helpful.
Bought an iPhone from the seller and everything was as described. We met in town and I checked the phone before paying.
Good service overall. They had the screen protector I was looking for and installed it for me.
The shop was busy when I arrived but the staff still assisted me. Prices were reasonable compared with other shops.
I bought a second-hand Samsung. The seller allowed me to test everything before making payment, which I appreciated.
Service was alright. I went there to replace my phone battery and collected it later the same day.
Friendly staff and the shop was clean. They explained the different data options before I decided.
I had to wait about 20 minutes because there were several people ahead of me, but my SIM registration was done properly
Seller was easy to communicate with. I checked the phone's IMEI and battery before paying.
I only needed a phone cover but the service was still good. The lady showed me different options.
Bought a new phone here. The staff explained the warranty and gave me the receipt. So far everything is working fine.
The phone looked exactly like the pictures. We agreed on a meeting place and the transaction went smoothly.
Decent shop with a fair selection of accessories. I got a USB-C cable and earphones.
My experience was okay. The queue was a bit long but the staff were polite.
I bought a used Huawei phone. It had a few scratches that were already shown in the pictures. No complaints.
Helpful staff. They assisted me with setting up my new SIM and explained the packages available.
Prices are not the cheapest but the service was good. I would probably come back for accessories.
Seller was responsive and answered my questions about the phone before I went to collect it.
Small shop but they had what I needed. The gentleman who helped me knew a lot about the phones.
I went in for a screen replacement. It took longer than expected but the phone is working properly now.
Bought a second-hand iPhone after testing Face ID, camera and speakers. Everything was working.
Brand new iPhones for the LOWEST PRICE IN NAMIBIA!!! No other seller can beat these prices!!! 🔥🔥🔥
BEST SHOP EVER!!! PERFECT SERVICE!!! PERFECT PHONES!!! PERFECT PRICES!!! ⭐⭐⭐⭐⭐
Yoh these guys are doing the things hey. Got my iPhone same day and the price was just proper. Don't waste your time going around.
Honestly this shop is the plug in Windhoek. Phones are original, prices are lekker and the people know what they are doing.
Boss, this guy is 100% legit. I just sent the money and my phone came the next day. No stress at all.
Eish, I was scared at first but the seller came through big time. Phone is clean clean. You can buy from this person with your eyes closed.
These guys don't disappoint hey. Best prices in Namibia, no cap. I have bought three phones here already and everything is always perfect.
My brother recommended this place and wow, they really know their story. Good phones, good prices and friendly people.
Hai this place is on another level. I went there just to look but ended up buying a phone because the deal was too good.
I don't normally write reviews but this one deserves it. These people are honest and their service is next level.
Omunhu uuyamba? This shop really knows how to treat customers. I came with a small problem and they sorted me out just like that.
The service is lekker hey. I have never seen such affordable phones in Windhoek. These guys deserve more customers.
I was shocked by the price honestly. Other shops are charging an arm and a leg but here I got mine for a good deal.
Yoh, I can't complain at all. Everything was just smooth smooth from the beginning until I received my phone.

"""

# --- the code does the rest ---
csv_file = "namibian_reviews.csv"

# Load existing data (or start fresh if file is empty/new)
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
else:
    existing = pd.DataFrame(columns=["record_id","source","product_category","text",
                                     "label_annotator1","label_annotator2","final_label"])

# Split the pasted block into individual reviews (one per line)
new_lines = [line.strip() for line in raw_reviews.strip().split("\n") if line.strip()]

# Figure out the next record_id number
next_id = (existing["record_id"].max() + 1) if len(existing) > 0 else 1

# Build the new rows
new_rows = []
for i, review in enumerate(new_lines):
    new_rows.append({
        "record_id": int(next_id + i),
        "source": source,
        "product_category": category,
        "text": review,
        "label_annotator1": "",
        "label_annotator2": "",
        "final_label": ""
    })

# Add them and save
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(csv_file, index=False)

print(f"Added {len(new_rows)} reviews as category '{category}'.")
print(f"Total reviews now: {len(updated)}")

Added 36 reviews as category 'cell_shops'.
Total reviews now: 96


In [3]:
# ============================================================
# PASTE HELPER: turn raw reviews into clean CSV rows
# ============================================================
# HOW TO USE:
#  1. Set the category and source below
#  2. Paste your reviews between the triple quotes - ONE REVIEW PER LINE
#  3. Run the cell - it appends them to your CSV automatically,
#     handling all the commas, quotes and numbering for you.

import pandas as pd
import os

# ---- SETTINGS: change these for each batch ----
category = "cell_shops"     # what kind of business this batch is
source   = "Facebook_Marketplace"      # where you got them
# ------------------------------------------------

# Paste reviews here, ONE PER LINE, between the triple quotes.
# Don't worry about commas or quotes - just paste the raw text.
raw_reviews = """
This is the real plug in Namibia. If you need an iPhone just contact them and they will sort you out.
No long stories with this seller. You pay, they deliver. Simple as that. Very trustworthy person.
My phone was delivered all the way to Oshakati and everything was 100%. This seller is not playing games.
Guys please don't sleep on this seller. The prices are giving and the phones are legit. I will definitely buy again.
Namibia must support businesses like this one. My phone arrived looking brand new, even better than the pictures.
"""

# --- the code does the rest ---
csv_file = "namibian_reviews.csv"

# Load existing data (or start fresh if file is empty/new)
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
else:
    existing = pd.DataFrame(columns=["record_id","source","product_category","text",
                                     "label_annotator1","label_annotator2","final_label"])

# Split the pasted block into individual reviews (one per line)
new_lines = [line.strip() for line in raw_reviews.strip().split("\n") if line.strip()]

# Figure out the next record_id number
next_id = (existing["record_id"].max() + 1) if len(existing) > 0 else 1

# Build the new rows
new_rows = []
for i, review in enumerate(new_lines):
    new_rows.append({
        "record_id": int(next_id + i),
        "source": source,
        "product_category": category,
        "text": review,
        "label_annotator1": "",
        "label_annotator2": "",
        "final_label": ""
    })

# Add them and save
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(csv_file, index=False)

print(f"Added {len(new_rows)} reviews as category '{category}'.")
print(f"Total reviews now: {len(updated)}")

Added 5 reviews as category 'cell_shops'.
Total reviews now: 101


In [4]:
# ============================================================
# PASTE HELPER: turn raw reviews into clean CSV rows
# ============================================================
# HOW TO USE:
#  1. Set the category and source below
#  2. Paste your reviews between the triple quotes - ONE REVIEW PER LINE
#  3. Run the cell - it appends them to your CSV automatically,
#     handling all the commas, quotes and numbering for you.

import pandas as pd
import os

# ---- SETTINGS: change these for each batch ----
category = "car_dealership"     # what kind of business this batch is
source   = "Google_Maps"      #e you got them
# ------------------------------------------------

# Paste reviews here, ONE PER LINE, between the triple quotes.
# Don't worry about commas or quotes - just paste the raw text.
raw_reviews = """
Guys, this car is clean clean. Pictures don't even do it justice. Seller is 100% trustworthy.
Honestly, Namibia needs more dealerships like this. No stories, good cars and good people.
The deal was just lekker. Seller answered everything and the car was delivered without any problems
I told my friends about this place already. If you need a car, don't sleep on these guys.
Sharp sharp, everything was sorted quickly. Car is exactly what I wanted. I can recommend this seller any day.
I asked several questions about the car before viewing it. The seller answered them and allowed me to bring someone along for inspection.
Good experience overall. It took a few days to finish the paperwork but they kept me informed.
I viewed the car in person and checked the VIN before paying. The transaction was straightforward.
The dealership was clean and the staff were professional. Prices seemed similar to other dealerships I checked.
The salesperson was helpful but the vehicle I wanted was already sold. They showed me another similar model.
The seller was responsive but we could not agree on the price. No problems with the communication.
I bought a second-hand vehicle and received the necessary documents. So far the car has been reliable.
Good customer service. The vehicle was clean and ready for the test drive.
The seller was honest about the mileage and previous repairs. I decided not to buy because of the repairs needed.
Staff were professional. I was looking for a small family car and they showed me a few options.
The finance process took some time but the staff kept me updated.
I checked the vehicle with my mechanic before buying. The seller was okay with this.
Friendly service and no pressure. The test drive helped me make my decision.
The car was in reasonable condition and the description was mostly accurate. There were a few scratches that weren't mentioned.
The seller sent me additional pictures when I asked. I eventually viewed the vehicle before deciding not to buy.
I went in to compare two vehicles. The salesperson answered most of my questions and gave me the service history
Good selection of vehicles. The staff gave me time to look around without pressuring me.
I bought a used car after taking it for a test drive. The paperwork was completed without any major issues.
The salesperson was helpful and explained the financing options. I did not buy on the day but the experience was good.
I went to look at a used Toyota and the salesperson allowed me to inspect the car before discussing the price.
These guys are the plug.
No stress with this seller.
Proper deal, sharp sharp.
The car is clean hey.
Sharp service from these guy
Yoh, this deal is lekke
Boss came through properl
No stories, car is proper
The real car plug
These guys know cars
Lekker deal on the bakkie
Sharp sharp, deal was sorted
Yoh, this car is clean clean

"""

# --- the code does the rest ---
csv_file = "namibian_reviews.csv"

# Load existing data (or start fresh if file is empty/new)
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
else:
    existing = pd.DataFrame(columns=["record_id","source","product_category","text",
                                     "label_annotator1","label_annotator2","final_label"])

# Split the pasted block into individual reviews (one per line)
new_lines = [line.strip() for line in raw_reviews.strip().split("\n") if line.strip()]

# Figure out the next record_id number
next_id = (existing["record_id"].max() + 1) if len(existing) > 0 else 1

# Build the new rows
new_rows = []
for i, review in enumerate(new_lines):
    new_rows.append({
        "record_id": int(next_id + i),
        "source": source,
        "product_category": category,
        "text": review,
        "label_annotator1": "",
        "label_annotator2": "",
        "final_label": ""
    })

# Add them and save
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(csv_file, index=False)

print(f"Added {len(new_rows)} reviews as category '{category}'.")
print(f"Total reviews now: {len(updated)}")

Added 38 reviews as category 'car_dealership'.
Total reviews now: 139


In [5]:
# ============================================================
# PASTE HELPER: turn raw reviews into clean CSV rows
# ============================================================
# HOW TO USE:
#  1. Set the category and source below
#  2. Paste your reviews between the triple quotes - ONE REVIEW PER LINE
#  3. Run the cell - it appends them to your CSV automatically,
#     handling all the commas, quotes and numbering for you.

import pandas as pd
import os

# ---- SETTINGS: change these for each batch ----
category = "salon_beauty"   # what kind of business this batch is
source   = "Google_Maps"      #e you got them
# ------------------------------------------------

# Paste reviews here, ONE PER LINE, between the triple quotes.
# Don't worry about commas or quotes - just paste the raw text.
raw_reviews = """
Yoh, my hair is giving!
These ladies know their story
Hair came out lekker
Eish, they did the things!
My girl was not lying.
Proper braids, sharp sharp.
The plug for braids.
Hai, these ladies are good.
Hair is clean clean.
Lekker service from the girls.
They understood the assignment.
Yoh, I look proper now.
No stories, just good work.
Best salon in town hey
These ladies don't play.
"""

# --- the code does the rest ---
csv_file = "namibian_reviews.csv"

# Load existing data (or start fresh if file is empty/new)
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
else:
    existing = pd.DataFrame(columns=["record_id","source","product_category","text",
                                     "label_annotator1","label_annotator2","final_label"])

# Split the pasted block into individual reviews (one per line)
new_lines = [line.strip() for line in raw_reviews.strip().split("\n") if line.strip()]

# Figure out the next record_id number
next_id = (existing["record_id"].max() + 1) if len(existing) > 0 else 1

# Build the new rows
new_rows = []
for i, review in enumerate(new_lines):
    new_rows.append({
        "record_id": int(next_id + i),
        "source": source,
        "product_category": category,
        "text": review,
        "label_annotator1": "",
        "label_annotator2": "",
        "final_label": ""
    })

# Add them and save
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(csv_file, index=False)

print(f"Added {len(new_rows)} reviews as category '{category}'.")
print(f"Total reviews now: {len(updated)}")

Added 15 reviews as category 'salon_beauty'.
Total reviews now: 154


In [6]:
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")

print("Total rows:", len(nam))
print()

# Any leftover placeholder rows?
placeholders = nam[nam['text'].str.contains("replace this|paste your", case=False, na=False)]
print("Leftover placeholder rows:", len(placeholders))
print()

# Spread across categories and sources
print("By category:")
print(nam['product_category'].value_counts())
print()
print("By source:")
print(nam['source'].value_counts())
print()

# Very short reviews that will get dropped
nam['word_count'] = nam['text'].str.split().str.len()
print("Reviews under 3 words (will be dropped):", (nam['word_count'] < 3).sum())
print()

# Any exact duplicate reviews?
print("Duplicate review texts:", nam['text'].duplicated().sum())

Total rows: 154

Leftover placeholder rows: 0

By category:
product_category
salon_beauty          45
cell_shops            41
car_dealership        38
mobile_phones         10
beauty                 9
car_dealerships        8
smart_shop_Namibia     3
Name: count, dtype: int64

By source:
source
google_maps             96
Google_Maps             53
Facebook_Marketplace     5
Name: count, dtype: int64

Reviews under 3 words (will be dropped): 2

Duplicate review texts: 0


In [7]:
# ============================================================
# CLEANUP: standardise category and source names
# ============================================================
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")

# --- Fix categories: merge duplicates/variants into clean names ---
category_map = {
    "mobile_phones": "cell_shops",
    "beauty": "salon_beauty",
    "car_dealerships": "car_dealership",
    "smart_shop_Namibia": "cell_shops",   # Smart Shop sells phones/electronics
}
nam["product_category"] = nam["product_category"].replace(category_map)

# --- Fix sources: make capitalisation consistent ---
source_map = {
    "Google_Maps": "google_maps",
    "Facebook_Marketplace": "facebook_marketplace",
}
nam["source"] = nam["source"].replace(source_map)

# Save the cleaned version
nam.to_csv("namibian_reviews.csv", index=False)

# Show the cleaned result
print("Cleaned categories:")
print(nam["product_category"].value_counts())
print()
print("Cleaned sources:")
print(nam["source"].value_counts())

Cleaned categories:
product_category
salon_beauty      54
cell_shops        54
car_dealership    46
Name: count, dtype: int64

Cleaned sources:
source
google_maps             149
facebook_marketplace      5
Name: count, dtype: int64


In [8]:
# ============================================================
# ONE-AT-A-TIME LABELLING HELPER
# ============================================================
# Shows each review on its own, clearly. You type 0 or 1 and press Enter.
# Type 's' to skip (decide later), or 'q' to stop and save your progress.

import pandas as pd

nam = pd.read_csv("namibian_reviews.csv")

# Make sure the label column exists and is treated as text so blanks stay blank
if "label_annotator1" not in nam.columns:
    nam["label_annotator1"] = ""
nam["label_annotator1"] = nam["label_annotator1"].astype("object")

print("Labelling: type 0 = genuine, 1 = fake/suspicious, s = skip, q = quit & save")
print("=" * 65)

for i in range(len(nam)):
    # Skip ones you've already labelled (so you can resume anytime)
    current = nam.at[i, "label_annotator1"]
    if pd.notna(current) and str(current).strip() in ("0", "1"):
        continue

    # Show the review clearly
    print(f"\nReview {i+1} of {len(nam)}  |  Category: {nam.at[i,'product_category']}")
    print("-" * 65)
    print(nam.at[i, "text"])
    print("-" * 65)

    # Ask for your label
    answer = input("Your label (0/1/s/q): ").strip().lower()

    if answer == "q":
        print("\nStopping and saving your progress...")
        break
    elif answer == "s":
        continue
    elif answer in ("0", "1"):
        nam.at[i, "label_annotator1"] = answer
    else:
        print("  (not understood - skipping this one, you can redo it later)")
        continue

# Save progress
nam.to_csv("namibian_reviews.csv", index=False)

done = nam["label_annotator1"].isin(["0","1"]).sum()
print(f"\nSaved. You've labelled {done} of {len(nam)} so far.")

Labelling: type 0 = genuine, 1 = fake/suspicious, s = skip, q = quit & save

Review 1 of 154  |  Category: salon_beauty
-----------------------------------------------------------------
Great environment, good music and a lovely barber with great energy, artistic touch and great conversations. Definitely the recipe to an unforgettable haircut experience. Big ups Keanon
-----------------------------------------------------------------

Review 2 of 154  |  Category: salon_beauty
-----------------------------------------------------------------
I never experienced this from any other salon in Windhoek. Unprofessional and prices are over rated. My hair was completely damaged and I walked out with Elumen covered all over my skull and my shirt was damaged. During consultation they offered me one price and at payment an additional 1000 was added. So disappointed and pissed off.
-----------------------------------------------------------------

Review 3 of 154  |  Category: salon_beauty
------

In [9]:
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")

# How many you've labelled, and the split
labelled = nam["label_annotator1"].isin([0, 1, "0", "1"]).sum()
print("Total reviews:", len(nam))
print("Labelled by you (annotator1):", labelled)
print()

# Convert to numbers and show the genuine vs fake split
nam["label_annotator1"] = pd.to_numeric(nam["label_annotator1"], errors="coerce")
print("Your label split:")
print(nam["label_annotator1"].value_counts(dropna=False))
print("  (0 = genuine, 1 = fake/suspicious, NaN = unlabelled)")

Total reviews: 154
Labelled by you (annotator1): 154

Your label split:
label_annotator1
1    88
0    66
Name: count, dtype: int64
  (0 = genuine, 1 = fake/suspicious, NaN = unlabelled)


In [10]:
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")
nam["label_annotator1"] = pd.to_numeric(nam["label_annotator1"], errors="coerce")

print("=== 5 you marked FAKE (1) ===")
for t in nam[nam["label_annotator1"]==1]["text"].head(5):
    print("•", t[:150])
    print()

print("=== 5 you marked GENUINE (0) ===")
for t in nam[nam["label_annotator1"]==0]["text"].head(5):
    print("•", t[:150])
    print()

=== 5 you marked FAKE (1) ===
• I had a very disappointing experience at this salon. My hair was long and healthy. I booked for a cherry dark colour and a wolf cut but the result was

• I was very impressed with the level of service at this salon. The stylist listened to my requests and gave me exactly what I wanted. I will definitely

• I had a great experience with this mobile phone repair service. They were able to fix my phone quickly and at a reasonable price. The staff were frien

• The best tech services place I have been to.

• I got served so fast! Very professional and efficient!Thank you for helping me out! I’m forever grateful!

=== 5 you marked GENUINE (0) ===
• Great environment, good music and a lovely barber with great energy, artistic touch and great conversations. Definitely the recipe to an unforgettable

• I never experienced this from any other salon in Windhoek. Unprofessional and prices are over rated. My hair was completely damaged and I walked out w

• I was al

In [11]:
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")
for i, row in nam.iterrows():
    print(f"{row['record_id']}| {row['text']}")
    print()

1| Great environment, good music and a lovely barber with great energy, artistic touch and great conversations. Definitely the recipe to an unforgettable haircut experience. Big ups Keanon

2| I never experienced this from any other salon in Windhoek. Unprofessional and prices are over rated. My hair was completely damaged and I walked out with Elumen covered all over my skull and my shirt was damaged. During consultation they offered me one price and at payment an additional 1000 was added. So disappointed and pissed off.

3| I had a very disappointing experience at this salon. My hair was long and healthy. I booked for a cherry dark colour and a wolf cut but the result was not the colour requested and my hair was left damaged. A Brazilian keratin treatment was applied without my consent. I was told the service would cost around 1800 to 2000 but at checkout I was charged 5200. Extremely unhappy with the outcome.

4| I was also not provided with proof of payment. The overall experience

In [12]:
# ============================================================
# SEPARATE REAL (collected) FROM AI-GENERATED reviews
# ============================================================
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")

if "data_origin" not in nam.columns:
    nam["data_origin"] = ""
nam["data_origin"] = nam["data_origin"].astype("object")

print("For each review: type  r = real (I collected it),  a = AI-generated,  q = quit & save")
print("=" * 65)

for i in range(len(nam)):
    if str(nam.at[i,"data_origin"]).strip() in ("real","ai"):
        continue  # already tagged, skip (lets you resume)
    print(f"\nReview {nam.at[i,'record_id']}  |  {nam.at[i,'product_category']}")
    print("-"*65)
    print(nam.at[i,"text"][:300])
    print("-"*65)
    ans = input("real or AI? (r/a/q): ").strip().lower()
    if ans == "q":
        break
    elif ans == "r":
        nam.at[i,"data_origin"] = "real"
    elif ans == "a":
        nam.at[i,"data_origin"] = "ai"
    else:
        continue

nam.to_csv("namibian_reviews.csv", index=False)
real = (nam["data_origin"]=="real").sum()
ai = (nam["data_origin"]=="ai").sum()
print(f"\nSaved. Real: {real} | AI: {ai} | Untagged: {len(nam)-real-ai}")

For each review: type  r = real (I collected it),  a = AI-generated,  q = quit & save

Review 1  |  salon_beauty
-----------------------------------------------------------------
Great environment, good music and a lovely barber with great energy, artistic touch and great conversations. Definitely the recipe to an unforgettable haircut experience. Big ups Keanon
-----------------------------------------------------------------

Review 2  |  salon_beauty
-----------------------------------------------------------------
I never experienced this from any other salon in Windhoek. Unprofessional and prices are over rated. My hair was completely damaged and I walked out with Elumen covered all over my skull and my shirt was damaged. During consultation they offered me one price and at payment an additional 1000 was add
-----------------------------------------------------------------

Review 3  |  salon_beauty
-----------------------------------------------------------------
I had a very dis

In [13]:
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")

print("Data origin breakdown:")
print(nam["data_origin"].value_counts())
print()

# Keep only the real ones
real = nam[nam["data_origin"] == "real"].copy()
print("Real reviews to keep:", len(real))
print()

# How they spread across categories
print("Real reviews by category:")
print(real["product_category"].value_counts())
print()

# Check length (very short ones get dropped later)
real["word_count"] = real["text"].str.split().str.len()
print("Real reviews under 3 words (will be dropped):", (real["word_count"] < 3).sum())

Data origin breakdown:
data_origin
real    78
ai      76
Name: count, dtype: int64

Real reviews to keep: 78

Real reviews by category:
product_category
cell_shops        36
salon_beauty      23
car_dealership    19
Name: count, dtype: int64

Real reviews under 3 words (will be dropped): 1


In [14]:
# ============================================================
# SAVE CLEAN DATASET: real reviews only
# ============================================================
import pandas as pd
nam = pd.read_csv("namibian_reviews.csv")

# Keep only real reviews
real = nam[nam["data_origin"] == "real"].copy()

# Drop the 1 review under 3 words (too short to analyse)
real["word_count"] = real["text"].str.split().str.len()
real = real[real["word_count"] >= 3].copy()

# Renumber cleanly from 1
real = real.reset_index(drop=True)
real["record_id"] = range(1, len(real) + 1)

# Reset labels so you re-label with the corrected understanding
real["label_annotator1"] = ""
real["label_annotator2"] = ""
real["final_label"] = ""

# Keep only the columns we need, drop helper columns
keep_cols = ["record_id", "source", "product_category", "text",
             "label_annotator1", "label_annotator2", "final_label"]
real = real[keep_cols]

# Save as the clean working dataset
real.to_csv("namibian_reviews_clean.csv", index=False)
print(f"Saved 'namibian_reviews_clean.csv' with {len(real)} real reviews.")
print("This is now your honest, collected dataset.")

Saved 'namibian_reviews_clean.csv' with 77 real reviews.
This is now your honest, collected dataset.


In [15]:
# ============================================================
# PASTE HELPER: add REAL reviews to the clean dataset
# ============================================================
# HOW TO USE:
#  1. Set the category and source below
#  2. Paste your reviews between the triple quotes - ONE PER LINE
#  3. Run the cell - it appends them to the CLEAN csv automatically

import pandas as pd
import os

# ---- SETTINGS: change these for each batch ----
category = "cell_shops"   # salon_beauty / cell_shops / car_dealership
source   = "google_maps"    # keep lowercase so sources stay consistent
# ------------------------------------------------

# Paste reviews here, ONE PER LINE, between the triple quotes.
raw_reviews = """
Poor customer service. Very rude.  No manners
Very good customer service and quality products.
I salute all professional, thumbs up 👍reliable technician. …
Responsiveness, Quality, Professionalism, Value
Friendly and clean, customer service is great at oshakati Oshana mall branch is super.
Thanks for bringing my iPhone 15 pro max back to life after water damage.
These guys knows what they are doing I'm so happy with they're services,thank 
I really appreciate their work and your dedication to your profession is inspiring 🙏 thank you 🙏 …
Best prices in town and quick service! Highly recommended
These guys knows what they are doing I'm so happy with they're services,thank you
I fixed my phone screen 3 months ago ,until now I'm still happy ......you guys are the best!!!
I got my phone from this shop im so satisfied thank you guys
Great service and customer care❤️
Fast a reliable. Global tech has been my number one tech company.  I have had them fix multiple gadgets of mine and people I’ve recommended and never have I been disappointed by them.Been using them since they started probably 2018 and I still use their services. I’d highly recommend their great service 👌🏾
Excellent service! Very professional, fast, and reliable. They fixed my phone perfectly and provided great customer service. Highly recommended!
Very disappointing service. Reading their terms and conditions they are in breach. Completely unresponsive after months of waiting for updates on my belongings that I submitted for assessment and repair. Waited more than three days to get a response or update after payment was made in full. I had to message and call multiple times before getting a single answer. Now my inquiries go completely unanswered. I want my items back! This is not acceptable!
When it comes to Global Tech you can never go wrong. They provide you with the best service and quality products .
Good quality products, great service.
You keep canceling appointments with me without any explanation or call back.
Fake screens. Incompetence. Overpricing. I DO NOT RECOMEND


"""

# --- the code does the rest ---
csv_file = "namibian_reviews_clean.csv"   # <-- goes into your CLEAN file

# Load existing data (or start fresh if file is new)
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
else:
    existing = pd.DataFrame(columns=["record_id","source","product_category","text",
                                     "label_annotator1","label_annotator2","final_label"])

# Split the pasted block into individual reviews (one per line)
new_lines = [line.strip() for line in raw_reviews.strip().split("\n") if line.strip()]

# Figure out the next record_id number
next_id = (existing["record_id"].max() + 1) if len(existing) > 0 else 1

# Build the new rows
new_rows = []
for i, review in enumerate(new_lines):
    new_rows.append({
        "record_id": int(next_id + i),
        "source": source,
        "product_category": category,
        "text": review,
        "label_annotator1": "",
        "label_annotator2": "",
        "final_label": ""
    })

# Add them and save
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(csv_file, index=False)

print(f"Added {len(new_rows)} reviews as category '{category}'.")
print(f"Total reviews now: {len(updated)}")

Added 20 reviews as category 'cell_shops'.
Total reviews now: 97


In [16]:
# ============================================================
# PASTE HELPER: add REAL reviews to the clean dataset
# ============================================================
# HOW TO USE:
#  1. Set the category and source below
#  2. Paste your reviews between the triple quotes - ONE PER LINE
#  3. Run the cell - it appends them to the CLEAN csv automatically

import pandas as pd
import os

# ---- SETTINGS: change these for each batch ----
category = "car_dealership"   # salon_beauty / cell_shops / car_dealership
source   = "google_maps"    # keep lowercase so sources stay consistent
# ------------------------------------------------

# Paste reviews here, ONE PER LINE, between the triple quotes.
raw_reviews = """
Absolutely shocking! From software malfunction to being asked who is your service assistant????. None of the clients there knew!!! Booking stuff ups to quoting N$1,355 in labour to replace a clip! A 15 minute job. Very disappointing to the extent of looking at buying another make vehicle!
Friendly staff and nice atmosphere. State of the art showroom and a wide range of cars to choose from.
Service is on point. I always leave - a happy customer.
Nobody answers the phone, tried calling almost 5 times. Terrible
Trying the last 2 hours to get hold of Novel Mazda, no number works
Wide range of cars and friendly staff
Service is very poor
Good honest cars, good service, good prices !!
Price is not the same in shop as on website.
Second hand cars are plenty there
Very professional and friendly service
I have to call Toyota SA and log  a Formal Complain about how poor the phone service is at Pupkewitz Toyota .
Picked up my bakkie there best service, clean show room. Best experience ever. Hansa is the best dealer I worked with so far. But I still wanted them to give me a Diary book and face mask ☺️
One of the worst Customer service.  I called once and put on hold 5 times and then call ended.  Called once more, 2 holds.  Only on the third call was I able to get some sort of service.  For a dealership that prides itself on service delivery, it certainly lacks any.
Absolutely no updates. They fail to provide information on your vehicle's status.
Surely the best vehicle dealer in the country. Always a pleasure to deal with this team.
Great car deals, great service and people is friendly
Premium quality cars at an affordable price. - big ups on the location too😁👌 …
"""

# --- the code does the rest ---
csv_file = "namibian_reviews_clean.csv"   # <-- goes into your CLEAN file

# Load existing data (or start fresh if file is new)
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
else:
    existing = pd.DataFrame(columns=["record_id","source","product_category","text",
                                     "label_annotator1","label_annotator2","final_label"])

# Split the pasted block into individual reviews (one per line)
new_lines = [line.strip() for line in raw_reviews.strip().split("\n") if line.strip()]

# Figure out the next record_id number
next_id = (existing["record_id"].max() + 1) if len(existing) > 0 else 1

# Build the new rows
new_rows = []
for i, review in enumerate(new_lines):
    new_rows.append({
        "record_id": int(next_id + i),
        "source": source,
        "product_category": category,
        "text": review,
        "label_annotator1": "",
        "label_annotator2": "",
        "final_label": ""
    })

# Add them and save
updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
updated.to_csv(csv_file, index=False)

print(f"Added {len(new_rows)} reviews as category '{category}'.")
print(f"Total reviews now: {len(updated)}")

Added 18 reviews as category 'car_dealership'.
Total reviews now: 115


In [17]:
import pandas as pd
nam = pd.read_csv("namibian_reviews_clean.csv")

print("Total reviews:", len(nam))
print()
print("By category:")
print(nam["product_category"].value_counts())
print()
print("By source:")
print(nam["source"].value_counts())
print()

# Any accidental leftovers or placeholders
ph = nam["text"].str.contains("paste |replace this", case=False, na=False).sum()
print("Placeholder/blank rows:", ph)

# Duplicates (same review pasted twice by accident)
print("Duplicate texts:", nam["text"].duplicated().sum())

# Very short ones that get dropped
nam["wc"] = nam["text"].str.split().str.len()
print("Under 3 words (will drop):", (nam["wc"] < 3).sum())

Total reviews: 115

By category:
product_category
cell_shops        56
car_dealership    36
salon_beauty      23
Name: count, dtype: int64

By source:
source
google_maps             113
facebook_marketplace      2
Name: count, dtype: int64

Placeholder/blank rows: 0
Duplicate texts: 0
Under 3 words (will drop): 0


In [18]:
# ============================================================
# LABEL the clean dataset, one review at a time
# ============================================================
import pandas as pd
nam = pd.read_csv("namibian_reviews_clean.csv")

if "label_annotator1" not in nam.columns:
    nam["label_annotator1"] = ""
nam["label_annotator1"] = nam["label_annotator1"].astype("object")

print("0 = genuine, 1 = suspected fake, s = skip, q = quit & save")
print("=" * 65)

for i in range(len(nam)):
    current = nam.at[i, "label_annotator1"]
    if pd.notna(current) and str(current).strip() in ("0", "1"):
        continue
    print(f"\nReview {i+1} of {len(nam)}  |  {nam.at[i,'product_category']}")
    print("-" * 65)
    print(nam.at[i, "text"])
    print("-" * 65)
    ans = input("Label (0/1/s/q): ").strip().lower()
    if ans == "q":
        break
    elif ans == "s":
        continue
    elif ans in ("0", "1"):
        nam.at[i, "label_annotator1"] = ans
    else:
        continue

nam.to_csv("namibian_reviews_clean.csv", index=False)
done = nam["label_annotator1"].isin(["0","1"]).sum()
print(f"\nSaved. Labelled {done} of {len(nam)}.")

0 = genuine, 1 = suspected fake, s = skip, q = quit & save

Review 1 of 115  |  salon_beauty
-----------------------------------------------------------------
Great environment, good music and a lovely barber with great energy, artistic touch and great conversations. Definitely the recipe to an unforgettable haircut experience. Big ups Keanon
-----------------------------------------------------------------

Review 2 of 115  |  salon_beauty
-----------------------------------------------------------------
I never experienced this from any other salon in Windhoek. Unprofessional and prices are over rated. My hair was completely damaged and I walked out with Elumen covered all over my skull and my shirt was damaged. During consultation they offered me one price and at payment an additional 1000 was added. So disappointed and pissed off.
-----------------------------------------------------------------

Review 3 of 115  |  salon_beauty
-----------------------------------------------------

In [19]:
import pandas as pd
nam = pd.read_csv("namibian_reviews_clean.csv")

labelled = nam["label_annotator1"].isin([0,1,"0","1"]).sum()
print("Total reviews:", len(nam))
print("Labelled:", labelled)
print()

nam["label_annotator1"] = pd.to_numeric(nam["label_annotator1"], errors="coerce")
print("Your label split:")
print(nam["label_annotator1"].value_counts(dropna=False))
print("  (0 = genuine, 1 = suspected fake, NaN = unlabelled)")
print()

# Split by category too, to see if anything looks off
print("Fake (1) count per category:")
print(nam[nam["label_annotator1"]==1]["product_category"].value_counts())

Total reviews: 115
Labelled: 115

Your label split:
label_annotator1
0    79
1    36
Name: count, dtype: int64
  (0 = genuine, 1 = suspected fake, NaN = unlabelled)

Fake (1) count per category:
product_category
cell_shops        17
car_dealership    10
salon_beauty       9
Name: count, dtype: int64


In [1]:
# ============================================================
# CREATE ANNOTATOR 2's BLIND COPY
# ============================================================
import pandas as pd
nam = pd.read_csv("namibian_reviews_clean.csv")

# Give annotator 2 only the review + category, NOT your labels
ann2 = nam[["record_id", "product_category", "text"]].copy()
ann2["label"] = ""   # blank column for them to fill with 0 or 1

ann2.to_csv("annotator2_labels.csv", index=False)
print("Created 'annotator2_labels.csv'")
print("Send this file to your second annotator.")
print("They fill the 'label' column: 0 = genuine, 1 = suspected fake. Then send it back.")

Created 'annotator2_labels.csv'
Send this file to your second annotator.
They fill the 'label' column: 0 = genuine, 1 = suspected fake. Then send it back.


In [ ]:
# ============================================================
# ENHANCED FEATURES (corrected, self-contained)
# ============================================================
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

SLANG_TERMS = ["lekker","sharp sharp","sharp","the plug","no cap","hey",
               "proper","bru","yoh","eish","giving","clean clean","boss"]

def engineered_features(text):
    t = str(text)
    words = t.split()
    letters = [c for c in t if c.isalpha()]
    cap_ratio = (sum(1 for c in letters if c.isupper())/len(letters)) if letters else 0
    low = t.lower()
    slang_count = sum(low.count(term) for term in SLANG_TERMS)
    sentiment = vader.polarity_scores(t)["compound"]
    return [len(words), len(t), t.count("!"), cap_ratio, slang_count, sentiment]

feature_names = ["n_words","n_chars","n_exclaim","cap_ratio","slang_count","sentiment"]

X_train_text_raw, X_test_text_raw, _, _ = train_test_split(
    df["text"], df["label"], test_size=0.2, stratify=df["label"], random_state=42
)

eng_train = np.array([engineered_features(t) for t in X_train_text_raw])
eng_test  = np.array([engineered_features(t) for t in X_test_text_raw])

scaler = StandardScaler()
eng_train_scaled = scaler.fit_transform(eng_train)
eng_test_scaled  = scaler.transform(eng_test)

X_train_combined = hstack([X_train, csr_matrix(eng_train_scaled)])
X_test_combined  = hstack([X_test,  csr_matrix(eng_test_scaled)])

print("TF-IDF only features:     ", X_train.shape[1])
print("Engineered features added:", eng_train.shape[1])
print("Combined total features:  ", X_train_combined.shape[1])
print("\nEngineered features for first training review:")
for name, val in zip(feature_names, eng_train[0]):
    print(f"  {name}: {val:.3f}")

In [ ]:
%pip install vaderSentiment

In [4]:
# ============================================================
# FULL PIPELINE IN ONE CELL: TF-IDF vs TF-IDF + engineered features
# ============================================================
import pandas as pd, numpy as np, re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix

# 1. Load the Ott dataset
url = "https://raw.githubusercontent.com/rajsiddarth/Text_Analytics/master/deceptive-opinion.csv"
df = pd.read_csv(url)
df["label"] = df["deceptive"].map({"deceptive": 1, "truthful": 0})

# 2. Clean text
def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"[^a-z\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()
df["clean_text"] = df["text"].apply(clean_text)

# 3. Split (cleaned text for TF-IDF, raw text for engineered features - same seed)
Xtr_txt, Xte_txt, ytr, yte = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, stratify=df["label"], random_state=42)
Xtr_raw, Xte_raw, _, _ = train_test_split(
    df["text"], df["label"], test_size=0.2, stratify=df["label"], random_state=42)

# 4. TF-IDF
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=5000)
Xtr_tfidf = vec.fit_transform(Xtr_txt)
Xte_tfidf = vec.transform(Xte_txt)

# 5. Engineered features (VADER sentiment, with a fallback if not installed)
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v = SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(str(t))["compound"]
except Exception:
    def sentiment(t):
        low = str(t).lower()
        pos = sum(w in low for w in ["good","great","best","love","perfect","amazing"])
        neg = sum(w in low for w in ["bad","worst","poor","terrible","scam","disappointed"])
        return (pos - neg) / 5.0

SLANG = ["lekker","sharp sharp","sharp","the plug","no cap","hey","proper","bru","yoh","eish","giving","clean clean","boss"]
def eng(t):
    t = str(t); words = t.split(); letters = [c for c in t if c.isalpha()]
    cap = sum(1 for c in letters if c.isupper())/len(letters) if letters else 0
    return [len(words), len(t), t.count("!"), cap, sum(t.lower().count(s) for s in SLANG), sentiment(t)]

eng_tr = np.array([eng(t) for t in Xtr_raw])
eng_te = np.array([eng(t) for t in Xte_raw])
sc = MinMaxScaler()
eng_tr_s = sc.fit_transform(eng_tr)
eng_te_s = sc.transform(eng_te)

# 6. Combine TF-IDF with engineered features
Xtr_comb = hstack([Xtr_tfidf, csr_matrix(eng_tr_s)])
Xte_comb = hstack([Xte_tfidf, csr_matrix(eng_te_s)])

# 7. Train each model on BOTH feature sets and compare
def evaluate(Xtr, Xte, setname):
    out = []
    for mname, model in [("Logistic Regression", LogisticRegression(max_iter=1000)),
                         ("Random Forest", RandomForestClassifier(n_estimators=200, random_state=42)),
                         ("Naive Bayes", MultinomialNB())]:
        model.fit(Xtr, ytr)
        p = model.predict(Xte)
        out.append([setname, mname, round(accuracy_score(yte,p),3), round(precision_score(yte,p),3),
                    round(recall_score(yte,p),3), round(f1_score(yte,p),3)])
    return out

rows = evaluate(Xtr_tfidf, Xte_tfidf, "TF-IDF only") + evaluate(Xtr_comb, Xte_comb, "TF-IDF + features")
results_df = pd.DataFrame(rows, columns=["Feature set","Model","Accuracy","Precision","Recall","F1"])
print(results_df.to_string(index=False))

      Feature set               Model  Accuracy  Precision  Recall    F1
      TF-IDF only Logistic Regression     0.875      0.870   0.881 0.876
      TF-IDF only       Random Forest     0.856      0.856   0.856 0.856
      TF-IDF only         Naive Bayes     0.887      0.860   0.925 0.892
TF-IDF + features Logistic Regression     0.863      0.858   0.869 0.863
TF-IDF + features       Random Forest     0.850      0.841   0.863 0.852
TF-IDF + features         Naive Bayes     0.881      0.863   0.906 0.884


In [5]:
# ============================================================
# NAMIBIAN DATA TEST: do slang/style features help on local text?
# 5-fold cross-validation (fairer than one split on a small dataset)
# ============================================================
import pandas as pd, numpy as np, re
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix

# Load clean Namibian data; use annotator-1 labels for now
nam = pd.read_csv("namibian_reviews_clean.csv")
nam["y"] = pd.to_numeric(nam["label_annotator1"], errors="coerce")
nam = nam.dropna(subset=["y"]).reset_index(drop=True)
nam["y"] = nam["y"].astype(int)
print("Namibian reviews used:", len(nam),
      "| genuine:", (nam["y"]==0).sum(), " fake:", (nam["y"]==1).sum())

texts = nam["text"].astype(str).values
y = nam["y"].values

def clean_text(t):
    t = str(t).lower(); t = re.sub(r"[^a-z\s]", " ", t); return re.sub(r"\s+", " ", t).strip()

SLANG = ["lekker","sharp sharp","sharp","the plug","no cap","hey","proper","bru","yoh","eish","giving","clean clean","boss"]
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v = SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(str(t))["compound"]
except Exception:
    def sentiment(t):
        low = str(t).lower()
        return (sum(w in low for w in ["good","great","best","love","perfect","amazing"])
                - sum(w in low for w in ["bad","worst","poor","terrible","scam","disappointed"]))/5.0
def eng(t):
    t=str(t); w=t.split(); L=[c for c in t if c.isalpha()]
    cap = sum(1 for c in L if c.isupper())/len(L) if L else 0
    return [len(w), len(t), t.count("!"), cap, sum(t.lower().count(s) for s in SLANG), sentiment(t)]

models = {"Logistic Regression": lambda: LogisticRegression(max_iter=1000),
          "Random Forest": lambda: RandomForestClassifier(n_estimators=200, random_state=42),
          "Naive Bayes": lambda: MultinomialNB()}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run(use_features):
    s = {m: {"acc":[],"prec":[],"rec":[],"f1":[]} for m in models}
    for tr, te in skf.split(texts, y):
        vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=2000)
        Xtr = vec.fit_transform([clean_text(t) for t in texts[tr]])
        Xte = vec.transform([clean_text(t) for t in texts[te]])
        if use_features:
            etr = np.array([eng(t) for t in texts[tr]]); ete = np.array([eng(t) for t in texts[te]])
            scal = MinMaxScaler(); etr = scal.fit_transform(etr); ete = scal.transform(ete)
            Xtr = hstack([Xtr, csr_matrix(etr)]); Xte = hstack([Xte, csr_matrix(ete)])
        for m, make in models.items():
            md = make(); md.fit(Xtr, y[tr]); p = md.predict(Xte)
            s[m]["acc"].append(accuracy_score(y[te],p))
            s[m]["prec"].append(precision_score(y[te],p,zero_division=0))
            s[m]["rec"].append(recall_score(y[te],p,zero_division=0))
            s[m]["f1"].append(f1_score(y[te],p,zero_division=0))
    return s

rows = []
for setname, uf in [("TF-IDF only", False), ("TF-IDF + features", True)]:
    s = run(uf)
    for m in models:
        rows.append([setname, m, round(np.mean(s[m]["acc"]),3), round(np.mean(s[m]["prec"]),3),
                     round(np.mean(s[m]["rec"]),3), round(np.mean(s[m]["f1"]),3)])
res = pd.DataFrame(rows, columns=["Feature set","Model","Accuracy","Precision","Recall","F1"])
print("\n5-fold cross-validation on Namibian data (mean scores):")
print(res.to_string(index=False))

Namibian reviews used: 115 | genuine: 79  fake: 36

5-fold cross-validation on Namibian data (mean scores):
      Feature set               Model  Accuracy  Precision  Recall    F1
      TF-IDF only Logistic Regression     0.687      0.400   0.057 0.100
      TF-IDF only       Random Forest     0.661      0.333   0.086 0.130
      TF-IDF only         Naive Bayes     0.670      0.400   0.057 0.100
TF-IDF + features Logistic Regression     0.678      0.400   0.057 0.100
TF-IDF + features       Random Forest     0.687      0.467   0.143 0.218
TF-IDF + features         Naive Bayes     0.678      0.400   0.057 0.100


In [6]:
# ============================================================
# NAMIBIAN DATA with CLASS BALANCING (handles the 79/36 imbalance)
# ============================================================
import pandas as pd, numpy as np, re
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix

nam = pd.read_csv("namibian_reviews_clean.csv")
nam["y"] = pd.to_numeric(nam["label_annotator1"], errors="coerce")
nam = nam.dropna(subset=["y"]).reset_index(drop=True)
nam["y"] = nam["y"].astype(int)
texts = nam["text"].astype(str).values
y = nam["y"].values

def clean_text(t):
    t=str(t).lower(); t=re.sub(r"[^a-z\s]"," ",t); return re.sub(r"\s+"," ",t).strip()

SLANG = ["lekker","sharp sharp","sharp","the plug","no cap","hey","proper","bru","yoh","eish","giving","clean clean","boss"]
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v = SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(str(t))["compound"]
except Exception:
    def sentiment(t):
        low=str(t).lower()
        return (sum(w in low for w in ["good","great","best","love","perfect","amazing"])
                - sum(w in low for w in ["bad","worst","poor","terrible","scam","disappointed"]))/5.0
def eng(t):
    t=str(t); w=t.split(); L=[c for c in t if c.isalpha()]
    cap=sum(1 for c in L if c.isupper())/len(L) if L else 0
    return [len(w),len(t),t.count("!"),cap,sum(t.lower().count(s) for s in SLANG),sentiment(t)]

# Models WITH class balancing for LR and RF
models = {"Logistic Regression": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
          "Random Forest": lambda: RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
          "Naive Bayes": lambda: MultinomialNB()}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run(use_features):
    s = {m: {"acc":[],"prec":[],"rec":[],"f1":[]} for m in models}
    for tr, te in skf.split(texts, y):
        vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=2000)
        Xtr = vec.fit_transform([clean_text(t) for t in texts[tr]])
        Xte = vec.transform([clean_text(t) for t in texts[te]])
        if use_features:
            etr=np.array([eng(t) for t in texts[tr]]); ete=np.array([eng(t) for t in texts[te]])
            sc=MinMaxScaler(); etr=sc.fit_transform(etr); ete=sc.transform(ete)
            Xtr=hstack([Xtr,csr_matrix(etr)]); Xte=hstack([Xte,csr_matrix(ete)])
        for m, make in models.items():
            md=make(); md.fit(Xtr,y[tr]); p=md.predict(Xte)
            s[m]["acc"].append(accuracy_score(y[te],p))
            s[m]["prec"].append(precision_score(y[te],p,zero_division=0))
            s[m]["rec"].append(recall_score(y[te],p,zero_division=0))
            s[m]["f1"].append(f1_score(y[te],p,zero_division=0))
    return s

rows=[]
for setname, uf in [("TF-IDF only (balanced)", False), ("TF-IDF + features (balanced)", True)]:
    s=run(uf)
    for m in models:
        rows.append([setname,m,round(np.mean(s[m]["acc"]),3),round(np.mean(s[m]["prec"]),3),
                     round(np.mean(s[m]["rec"]),3),round(np.mean(s[m]["f1"]),3)])
res=pd.DataFrame(rows,columns=["Feature set","Model","Accuracy","Precision","Recall","F1"])
print("5-fold CV on Namibian data WITH class balancing (mean scores):")
print(res.to_string(index=False))

5-fold CV on Namibian data WITH class balancing (mean scores):
                 Feature set               Model  Accuracy  Precision  Recall    F1
      TF-IDF only (balanced) Logistic Regression     0.704      0.542   0.439 0.482
      TF-IDF only (balanced)       Random Forest     0.713      0.636   0.307 0.395
      TF-IDF only (balanced)         Naive Bayes     0.670      0.400   0.057 0.100
TF-IDF + features (balanced) Logistic Regression     0.696      0.528   0.525 0.514
TF-IDF + features (balanced)       Random Forest     0.696      0.547   0.250 0.338
TF-IDF + features (balanced)         Naive Bayes     0.678      0.400   0.057 0.100


In [7]:
# ============================================================
# LIVE DEMO: final model + predict any review on the spot
# ============================================================
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix
import joblib

nam = pd.read_csv("namibian_reviews_clean.csv")
nam["y"] = pd.to_numeric(nam["label_annotator1"], errors="coerce")
nam = nam.dropna(subset=["y"]).reset_index(drop=True).astype({"y":int})
texts = nam["text"].astype(str).values
y = nam["y"].values

def clean_text(t):
    t=str(t).lower(); t=re.sub(r"[^a-z\s]"," ",t); return re.sub(r"\s+"," ",t).strip()
SLANG=["lekker","sharp sharp","sharp","the plug","no cap","hey","proper","bru","yoh","eish","giving","clean clean","boss"]
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v=SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(str(t))["compound"]
except Exception:
    def sentiment(t):
        low=str(t).lower()
        return (sum(w in low for w in ["good","great","best","love","perfect","amazing"])
                - sum(w in low for w in ["bad","worst","poor","terrible","scam","disappointed"]))/5.0
def eng(t):
    t=str(t); w=t.split(); L=[c for c in t if c.isalpha()]
    cap=sum(1 for c in L if c.isupper())/len(L) if L else 0
    return [len(w),len(t),t.count("!"),cap,sum(t.lower().count(s) for s in SLANG),sentiment(t)]

# Train the FINAL model on ALL data
final_vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=2000)
Xtfidf = final_vec.fit_transform([clean_text(t) for t in texts])
final_scaler = MinMaxScaler()
Xeng = final_scaler.fit_transform(np.array([eng(t) for t in texts]))
Xall = hstack([Xtfidf, csr_matrix(Xeng)])
final_model = LogisticRegression(max_iter=1000, class_weight="balanced")
final_model.fit(Xall, y)

joblib.dump(final_model, "namibian_final_model.pkl")
joblib.dump(final_vec, "namibian_vectorizer.pkl")
joblib.dump(final_scaler, "namibian_scaler.pkl")
print("Final model trained and saved.\n")

def predict_review(text):
    xt = final_vec.transform([clean_text(text)])
    xe = final_scaler.transform([eng(text)])
    X = hstack([xt, csr_matrix(xe)])
    pred = final_model.predict(X)[0]
    prob_fake = final_model.predict_proba(X)[0][1]
    label = "LIKELY FAKE" if pred == 1 else "LIKELY GENUINE"
    print(f'Review: "{text}"')
    print(f"Prediction: {label}  (estimated fake probability: {prob_fake:.0%})\n")
    return label

# Try a couple
predict_review("best salon ever!!! amazing amazing highly recommend!!!")
predict_review("I waited three days and the seller kept me updated, the phone works fine")

Final model trained and saved.

Review: "best salon ever!!! amazing amazing highly recommend!!!"
Prediction: LIKELY FAKE  (estimated fake probability: 73%)

Review: "I waited three days and the seller kept me updated, the phone works fine"
Prediction: LIKELY GENUINE  (estimated fake probability: 35%)



'LIKELY GENUINE'

In [8]:
# ============================================================
# DECEPTIVE INDICATORS: what signals fake vs genuine in Namibian reviews
# (uses the final LR model trained on all data - answers objective SO4)
# ============================================================
import numpy as np

# Feature names: the TF-IDF words, then our 6 engineered features
tfidf_names = list(final_vec.get_feature_names_out())
eng_names = ["word_count", "char_count", "exclamations", "caps_ratio", "slang_count", "sentiment"]
all_names = np.array(tfidf_names + eng_names)

# The model's learned weights (positive = pushes toward FAKE, negative = GENUINE)
coefs = final_model.coef_[0]

order = np.argsort(coefs)
top_fake = order[-15:][::-1]      # most positive = most fake-leaning
top_genuine = order[:15]          # most negative = most genuine-leaning

print("=" * 55)
print("TOP INDICATORS OF A *FAKE* REVIEW (Namibian data)")
print("=" * 55)
for i in top_fake:
    tag = " [engineered]" if all_names[i] in eng_names else ""
    print(f"  {all_names[i]:20s}  {coefs[i]:+.3f}{tag}")

print("\n" + "=" * 55)
print("TOP INDICATORS OF A *GENUINE* REVIEW (Namibian data)")
print("=" * 55)
for i in top_genuine:
    tag = " [engineered]" if all_names[i] in eng_names else ""
    print(f"  {all_names[i]:20s}  {coefs[i]:+.3f}{tag}")

# Where did the 6 engineered features rank overall?
print("\n" + "=" * 55)
print("YOUR ENGINEERED FEATURES - weight and direction")
print("=" * 55)
for name in eng_names:
    idx = list(all_names).index(name)
    direction = "→ FAKE" if coefs[idx] > 0 else "→ GENUINE"
    print(f"  {name:15s}  {coefs[idx]:+.3f}  {direction}")

TOP INDICATORS OF A *FAKE* REVIEW (Namibian data)
  sentiment             +0.966 [engineered]
  these                 +0.793
  always                +0.591
  these ladies          +0.570
  ladies                +0.570
  quality               +0.555
  helpful               +0.539
  people                +0.522
  great service         +0.475
  exclamations          +0.474 [engineered]
  place                 +0.451
  here                  +0.429
  this                  +0.417
  service               +0.414
  sim                   +0.387

TOP INDICATORS OF A *GENUINE* REVIEW (Namibian data)
  char_count            -0.828 [engineered]
  word_count            -0.793 [engineered]
  not                   -0.626
  phone                 -0.487
  your                  -0.454
  but the               -0.396
  at                    -0.385
  an                    -0.373
  the                   -0.353
  the phone             -0.344
  poor                  -0.331
  after                 -0.329
  it   

In [9]:
# ============================================================
# CREATE ANNOTATOR 2's BLIND FILE (for Google Sheets)
# ============================================================
import pandas as pd
nam = pd.read_csv("namibian_reviews_clean.csv")

# Only the columns annotator 2 should see - NOT your labels
ann2 = nam[["record_id", "product_category", "text"]].copy()
ann2["label"] = ""   # blank column: they put 0 or 1 here

ann2.to_csv("annotator2_labels.csv", index=False)
print("Created 'annotator2_labels.csv' with", len(ann2), "reviews, ready to upload.")

Created 'annotator2_labels.csv' with 115 reviews, ready to upload.


In [10]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# --- set this to the exact name of the file you downloaded ---
ANN2_FILE = "Thurston_annotator2.csv"   # change if you saved it under a different name

# your main file (has annotator 1's labels)
clean = pd.read_csv("namibian_reviews_clean.csv")

# annotator 2's downloaded file
ann2 = pd.read_csv(ANN2_FILE)
ann2 = ann2[["record_id", "label"]].rename(columns={"label": "label_annotator2"})

# merge annotator 2's labels onto the main file by record_id
merged = clean.drop(columns=["label_annotator2"], errors="ignore").merge(ann2, on="record_id", how="left")

for col in ["label_annotator1", "label_annotator2"]:
    merged[col] = pd.to_numeric(merged[col], errors="coerce")

missing = merged["label_annotator2"].isna().sum()
print("Rows total:", len(merged))
print("Annotator 2 labels missing:", missing)

if missing == 0:
    a1 = merged["label_annotator1"].astype(int)
    a2 = merged["label_annotator2"].astype(int)

    kappa = cohen_kappa_score(a1, a2)
    agree = (a1 == a2).sum()
    disagree = (a1 != a2).sum()

    print("\n--- Inter-annotator agreement ---")
    print(f"Agreed on:    {agree} / {len(merged)}  ({agree/len(merged)*100:.1f}%)")
    print(f"Disagreed on: {disagree}")
    print(f"Cohen's Kappa: {kappa:.3f}")

    print("\n--- Reviews where you two disagree ---")
    mask = (a1.values != a2.values)
    disagreements = merged[mask][["record_id","label_annotator1","label_annotator2","text"]]
    for _, r in disagreements.iterrows():
        print(f"\n[{r['record_id']}]  you={int(r['label_annotator1'])}  annotator2={int(r['label_annotator2'])}")
        print(f"   {r['text']}")
else:
    print("\nAnnotator 2 left some rows blank — ask him to fill those before we go on.")

FileNotFoundError: [Errno 2] No such file or directory: 'Thurston_annotator2.csv'

In [11]:
import os
print("CSV files I can see here:")
for f in os.listdir():
    if f.lower().endswith(".csv"):
        print("  ", repr(f))

CSV files I can see here:
   'annotator2_labels.csv'
   'namibian_reviews.csv'
   'namibian_reviews_clean.csv'
   'Thurston_annotator2.csv.csv'


In [12]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# --- set this to the exact name of the file you downloaded ---
ANN2_FILE = "Thurston_annotator2.csv.csv"   # change if you saved it under a different name

# your main file (has annotator 1's labels)
clean = pd.read_csv("namibian_reviews_clean.csv")

# annotator 2's downloaded file
ann2 = pd.read_csv(ANN2_FILE)
ann2 = ann2[["record_id", "label"]].rename(columns={"label": "label_annotator2"})

# merge annotator 2's labels onto the main file by record_id
merged = clean.drop(columns=["label_annotator2"], errors="ignore").merge(ann2, on="record_id", how="left")

for col in ["label_annotator1", "label_annotator2"]:
    merged[col] = pd.to_numeric(merged[col], errors="coerce")

missing = merged["label_annotator2"].isna().sum()
print("Rows total:", len(merged))
print("Annotator 2 labels missing:", missing)

if missing == 0:
    a1 = merged["label_annotator1"].astype(int)
    a2 = merged["label_annotator2"].astype(int)

    kappa = cohen_kappa_score(a1, a2)
    agree = (a1 == a2).sum()
    disagree = (a1 != a2).sum()

    print("\n--- Inter-annotator agreement ---")
    print(f"Agreed on:    {agree} / {len(merged)}  ({agree/len(merged)*100:.1f}%)")
    print(f"Disagreed on: {disagree}")
    print(f"Cohen's Kappa: {kappa:.3f}")

    print("\n--- Reviews where you two disagree ---")
    mask = (a1.values != a2.values)
    disagreements = merged[mask][["record_id","label_annotator1","label_annotator2","text"]]
    for _, r in disagreements.iterrows():
        print(f"\n[{r['record_id']}]  you={int(r['label_annotator1'])}  annotator2={int(r['label_annotator2'])}")
        print(f"   {r['text']}")
else:
    print("\nAnnotator 2 left some rows blank — ask him to fill those before we go on.")

Rows total: 115
Annotator 2 labels missing: 0

--- Inter-annotator agreement ---
Agreed on:    92 / 115  (80.0%)
Disagreed on: 23
Cohen's Kappa: 0.538

--- Reviews where you two disagree ---

[11]  you=0  annotator2=1
   I had a great experience with this mobile phone repair service. They were able to fix my phone quickly and at a reasonable price. The staff were friendly and knowledgeable, and I would definitely recommend them to others.

[21]  you=0  annotator2=1
   No fresh milk. Not out of stock, they literally don't stock fresh milk. I asked 2 different staff, they both only knew about the long life milk. They have a dairy shelf a mile long with every conceivable dairy product, but no milk. What kind of supermarket doesn't have fresh milk?

[34]  you=0  annotator2=1
   Staff were polite and the salon was clean. My hair treatment took around two hours.

[38]  you=1  annotator2=0
   I bought a charger here last week. The service was okay and the lady at the counter was helpful.

[44

In [13]:
import pandas as pd
a1 = pd.to_numeric(merged["label_annotator1"], errors="coerce").astype(int)
a2 = pd.to_numeric(merged["label_annotator2"], errors="coerce").astype(int)
dis = merged[a1.values != a2.values][["record_id","label_annotator1","label_annotator2","text"]].reset_index(drop=True)

pd.set_option("display.max_colwidth", None)
for i, r in dis.iterrows():
    print(f"{i+1}. [{r['record_id']}]  you={int(r['label_annotator1'])}  Thurston={int(r['label_annotator2'])}")
    print(f"   {r['text']}\n")

1. [11]  you=0  Thurston=1
   I had a great experience with this mobile phone repair service. They were able to fix my phone quickly and at a reasonable price. The staff were friendly and knowledgeable, and I would definitely recommend them to others.

2. [21]  you=0  Thurston=1
   No fresh milk. Not out of stock, they literally don't stock fresh milk. I asked 2 different staff, they both only knew about the long life milk. They have a dairy shelf a mile long with every conceivable dairy product, but no milk. What kind of supermarket doesn't have fresh milk?

3. [34]  you=0  Thurston=1
   Staff were polite and the salon was clean. My hair treatment took around two hours.

4. [38]  you=1  Thurston=0
   I bought a charger here last week. The service was okay and the lady at the counter was helpful.

5. [44]  you=1  Thurston=0
   I had to wait about 20 minutes because there were several people ahead of me, but my SIM registration was done properly

6. [47]  you=1  Thurston=0
   Bought a n

In [14]:
a1 = merged["label_annotator1"].astype(int)
a2 = merged["label_annotator2"].astype(int)
dis = merged[a1.values != a2.values]

print("# Copy this into a NEW cell and fill each one in with 0 or 1 (your final call)\n")
print("decisions = {")
for _, r in dis.iterrows():
    print(f'    {r["record_id"]!r}: None,   # {r["text"][:70]}...')
print("}")

# Copy this into a NEW cell and fill each one in with 0 or 1 (your final call)

decisions = {
    11: None,   # I had a great experience with this mobile phone repair service. They w...
    21: None,   # No fresh milk. Not out of stock, they literally don't stock fresh milk...
    34: None,   # Staff were polite and the salon was clean. My hair treatment took arou...
    38: None,   # I bought a charger here last week. The service was okay and the lady a...
    44: None,   # I had to wait about 20 minutes because there were several people ahead...
    47: None,   # Bought a new phone here. The staff explained the warranty and gave me ...
    54: None,   # Eish, I was scared at first but the seller came through big time. Phon...
    56: None,   # I don't normally write reviews but this one deserves it. These people ...
    60: None,   # My phone was delivered all the way to Oshakati and everything was 100%...
    61: None,   # Guys, this car is clean clean. Pictures don't even do it jus

In [15]:
# Copy this into a NEW cell and fill each one in with 0 or 1 (your final call)

decisions = {
    11: 1,   # I had a great experience with this mobile phone repair service. They w...
    21: 0,   # No fresh milk. Not out of stock, they literally don't stock fresh milk...
    34: 0,   # Staff were polite and the salon was clean. My hair treatment took arou...
    38: 0,   # I bought a charger here last week. The service was okay and the lady a...
    44: 0,   # I had to wait about 20 minutes because there were several people ahead...
    47: 0,   # Bought a new phone here. The staff explained the warranty and gave me ...
    54: 1,   # Eish, I was scared at first but the seller came through big time. Phon...
    56: 1,   # I don't normally write reviews but this one deserves it. These people ...
    60: 0,   # My phone was delivered all the way to Oshakati and everything was 100%...
    61: 1,   # Guys, this car is clean clean. Pictures don't even do it justice. Sell...
    64: 1,   # Good customer service. The vehicle was clean and ready for the test dr...
    72: 1,   # These ladies know their story...
    76: 0,   # Best salon in town hey...
    79: 0,   # Very good customer service and quality products....
    80: 0,   # I salute all professional, thumbs up 👍reliable technician. …...
    84: 1,   # These guys knows what they are doing I'm so happy with they're service...
    87: 0,   # These guys knows what they are doing I'm so happy with they're service...
    90: 0,   # Great service and customer care❤️...
    92: 1,   # Excellent service! Very professional, fast, and reliable. They fixed m...
    99: 1,   # Friendly staff and nice atmosphere. State of the art showroom and a wi...
    108: 0,   # Very professional and friendly service...
    113: 1,   # Surely the best vehicle dealer in the country. Always a pleasure to de...
    115: 1,   # Premium quality cars at an affordable price. - big ups on the location...
}

In [16]:
merged["final_label"] = merged["label_annotator1"].astype(int)
for rid, lab in decisions.items():
    merged.loc[merged["record_id"] == rid, "final_label"] = lab

# safety checks
assert merged["final_label"].isna().sum() == 0, "some rows still unlabelled"
assert all(v in (0,1) for v in decisions.values()), "every decision must be 0 or 1"

print("Final label counts:")
print(merged["final_label"].value_counts())
merged.to_csv("namibian_reviews_final.csv", index=False)
print("\nSaved -> namibian_reviews_final.csv")

Final label counts:
final_label
0    79
1    36
Name: count, dtype: int64

Saved -> namibian_reviews_final.csv


In [17]:
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.sparse import hstack, csr_matrix

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v = SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t): return 0.0

df = pd.read_csv("namibian_reviews_final.csv")
df["text"] = df["text"].astype(str)
y = df["final_label"].astype(int).values
raw = df["text"].values

SLANG = ["lekker","sharp sharp","sharp","the plug","no cap","hey","proper",
         "bru","yoh","eish","giving","clean clean","boss"]

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+"," ",t)
    t = re.sub(r"[^a-z0-9\s!]"," ",t)
    return re.sub(r"\s+"," ",t).strip()

def eng_features(texts):
    rows = []
    for t in texts:
        low, cc = t.lower(), len(t)
        rows.append([
            len(t.split()), cc, t.count("!"),
            (sum(1 for c in t if c.isupper())/cc if cc else 0),
            sum(low.count(s) for s in SLANG), sentiment(t)
        ])
    return np.array(rows, dtype=float)

clean = np.array([clean_text(t) for t in raw])

models = {
    "Logistic Regression": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest":       lambda: RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "Naive Bayes":         lambda: MultinomialNB(),
}

def run_cv(use_features):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    res = {m: {"acc":[],"prec":[],"rec":[],"f1":[]} for m in models}
    for tr, te in skf.split(clean, y):
        vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=2000)
        Xtr, Xte = vec.fit_transform(clean[tr]), vec.transform(clean[te])
        if use_features:
            sc = MinMaxScaler()
            Xtr = hstack([Xtr, csr_matrix(sc.fit_transform(eng_features(raw[tr])))])
            Xte = hstack([Xte, csr_matrix(sc.transform(eng_features(raw[te])))])
        for name, make in models.items():
            clf = make(); clf.fit(Xtr, y[tr]); p = clf.predict(Xte)
            res[name]["acc"].append(accuracy_score(y[te], p))
            res[name]["prec"].append(precision_score(y[te], p, zero_division=0))
            res[name]["rec"].append(recall_score(y[te], p, zero_division=0))
            res[name]["f1"].append(f1_score(y[te], p, zero_division=0))
    return res

def show(res, title):
    print(f"\n=== {title} ===")
    print(f"{'Model':22}{'Acc':>7}{'Prec':>7}{'Rec':>7}{'F1':>7}")
    for name, r in res.items():
        print(f"{name:22}{np.mean(r['acc']):7.3f}{np.mean(r['prec']):7.3f}"
              f"{np.mean(r['rec']):7.3f}{np.mean(r['f1']):7.3f}")

show(run_cv(False), "TF-IDF only")
show(run_cv(True),  "TF-IDF + engineered features")


=== TF-IDF only ===
Model                     Acc   Prec    Rec     F1
Logistic Regression     0.739  0.583  0.500  0.532
Random Forest           0.704  0.500  0.246  0.320
Naive Bayes             0.678  0.300  0.057  0.094

=== TF-IDF + engineered features ===
Model                     Acc   Prec    Rec     F1
Logistic Regression     0.704  0.529  0.500  0.513
Random Forest           0.678  0.287  0.143  0.184
Naive Bayes             0.678  0.200  0.029  0.050


In [18]:
import joblib, re, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("namibian_reviews_final.csv")
df["text"]=df["text"].astype(str)
y=df["final_label"].astype(int).values

def clean_text(t):
    t=t.lower(); t=re.sub(r"http\S+"," ",t)
    t=re.sub(r"[^a-z0-9\s!]"," ",t); return re.sub(r"\s+"," ",t).strip()

clean=[clean_text(t) for t in df["text"].values]
vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=2000)
X=vec.fit_transform(clean)
model=LogisticRegression(max_iter=1000,class_weight="balanced").fit(X,y)

joblib.dump(model,"demo_model.pkl")
joblib.dump(vec,"demo_vectorizer.pkl")
print("Saved -> demo_model.pkl, demo_vectorizer.pkl")

def predict_review(text):
    prob=model.predict_proba(vec.transform([clean_text(text)]))[0][1]
    verdict="LIKELY FAKE" if prob>=0.5 else "LIKELY GENUINE"
    print(f"\nReview: {text}\nVerdict: {verdict}  (fake probability {prob:.0%})")
    return prob

predict_review("BEST SHOP EVER!!! PERFECT PHONES!!! amazing prices!!!")
predict_review("I bought a charger here, it works fine and the staff gave me a receipt.")

Saved -> demo_model.pkl, demo_vectorizer.pkl

Review: BEST SHOP EVER!!! PERFECT PHONES!!! amazing prices!!!
Verdict: LIKELY GENUINE  (fake probability 49%)

Review: I bought a charger here, it works fine and the staff gave me a receipt.
Verdict: LIKELY GENUINE  (fake probability 44%)


np.float64(0.44187324520369015)

In [19]:
import joblib, re, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, csr_matrix

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v=SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t): return 0.0

df=pd.read_csv("namibian_reviews_final.csv"); df["text"]=df["text"].astype(str)
y=df["final_label"].astype(int).values; raw=df["text"].values

SLANG=["lekker","sharp sharp","sharp","the plug","no cap","hey","proper",
       "bru","yoh","eish","giving","clean clean","boss"]

def clean_text(t):
    t=t.lower(); t=re.sub(r"http\S+"," ",t)
    t=re.sub(r"[^a-z0-9\s!]"," ",t); return re.sub(r"\s+"," ",t).strip()

def eng_features(texts):
    rows=[]
    for t in texts:
        low,cc=t.lower(),len(t)
        rows.append([len(t.split()),cc,t.count("!"),
                     (sum(1 for c in t if c.isupper())/cc if cc else 0),
                     sum(low.count(s) for s in SLANG), sentiment(t)])
    return np.array(rows,dtype=float)

clean=[clean_text(t) for t in raw]
vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=2000)
X=vec.fit_transform(clean)
sc=MinMaxScaler(); Xf=hstack([X, csr_matrix(sc.fit_transform(eng_features(raw)))])
model=LogisticRegression(max_iter=1000,class_weight="balanced").fit(Xf,y)

joblib.dump(model,"demo_model.pkl")
joblib.dump(vec,"demo_vectorizer.pkl")
joblib.dump(sc,"demo_scaler.pkl")
print("Saved -> demo_model.pkl, demo_vectorizer.pkl, demo_scaler.pkl")

def predict_review(text):
    xf=hstack([vec.transform([clean_text(text)]),
               csr_matrix(sc.transform(eng_features([text])))])
    prob=float(model.predict_proba(xf)[0][1])
    verdict="LIKELY FAKE" if prob>=0.5 else "LIKELY GENUINE"
    print(f"\nReview: {text}\nVerdict: {verdict}  (fake probability {prob:.0%})")
    return prob

predict_review("BEST SHOP EVER!!! PERFECT PHONES!!! amazing prices!!!")
predict_review("I bought a charger here, it works fine and the staff gave me a receipt.")
predict_review("These guys are the plug, buy with your eyes closed sharp sharp")

Saved -> demo_model.pkl, demo_vectorizer.pkl, demo_scaler.pkl

Review: BEST SHOP EVER!!! PERFECT PHONES!!! amazing prices!!!
Verdict: LIKELY FAKE  (fake probability 64%)

Review: I bought a charger here, it works fine and the staff gave me a receipt.
Verdict: LIKELY GENUINE  (fake probability 45%)

Review: These guys are the plug, buy with your eyes closed sharp sharp
Verdict: LIKELY FAKE  (fake probability 59%)


0.5920466448770713

In [1]:
import sys
!"{sys.executable}" gui.py

c:\Users\Acer\AppData\Local\Python\pythoncore-3.14-64\python.exe: can't open file 'c:\\Users\\Acer\\Documents\\school\\Research Project\\Fake Review Detector\\gui.py': [Errno 2] No such file or directory


In [2]:
gui_code = r'''
import re
import tkinter as tk
from tkinter import scrolledtext, messagebox
import numpy as np
import joblib
from scipy.sparse import hstack, csr_matrix

SLANG = ["lekker", "sharp sharp", "sharp", "the plug", "no cap", "hey",
         "proper", "bru", "yoh", "eish", "giving", "clean clean", "boss"]

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _vader = SentimentIntensityAnalyzer()
    def sentiment(t):
        return _vader.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t):
        return 0.0

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z0-9\s!]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def eng_features(text):
    low, cc = text.lower(), len(text)
    return np.array([[
        len(text.split()), cc, text.count("!"),
        (sum(1 for c in text if c.isupper()) / cc if cc else 0),
        sum(low.count(s) for s in SLANG), sentiment(text),
    ]], dtype=float)

try:
    model = joblib.load("demo_model.pkl")
    vec = joblib.load("demo_vectorizer.pkl")
    scaler = joblib.load("demo_scaler.pkl")
except Exception as e:
    tk.Tk().withdraw()
    messagebox.showerror("Model files not found",
        "gui.py must be in the same folder as the .pkl files.\n\n" + str(e))
    raise

def analyse():
    text = input_box.get("1.0", "end").strip()
    if not text:
        result_var.set("Please paste a review first.")
        result_label.config(fg="black"); detail_var.set(""); return
    x = vec.transform([clean_text(text)])
    f = scaler.transform(eng_features(text))
    xf = hstack([x, csr_matrix(f)])
    prob = float(model.predict_proba(xf)[0][1])
    if prob >= 0.5:
        result_var.set(f"\u26a0  LIKELY FAKE    (fake probability {prob:.0%})")
        result_label.config(fg="#b00020")
    else:
        result_var.set(f"\u2714  LIKELY GENUINE    (fake probability {prob:.0%})")
        result_label.config(fg="#1b7a1b")
    fe = eng_features(text)[0]
    detail_var.set(
        f"words: {int(fe[0])}     characters: {int(fe[1])}     exclamations: {int(fe[2])}\n"
        f"capitals ratio: {fe[3]:.2f}     slang words: {int(fe[4])}     sentiment: {fe[5]:.2f}")

root = tk.Tk()
root.title("Fake Review Detector")
root.geometry("660x540")
root.configure(bg="white")
tk.Label(root, text="Fake Review Detector", font=("Segoe UI", 18, "bold"), bg="white").pack(pady=(18, 2))
tk.Label(root, text="Detecting deceptive Namibian online consumer reviews",
         font=("Segoe UI", 10), fg="#555", bg="white").pack(pady=(0, 14))
tk.Label(root, text="Paste a review:", font=("Segoe UI", 11), bg="white").pack(anchor="w", padx=26)
input_box = scrolledtext.ScrolledText(root, height=6, width=66, font=("Segoe UI", 11), wrap="word")
input_box.pack(padx=26, pady=(4, 10))
tk.Button(root, text="Analyse review", font=("Segoe UI", 12, "bold"),
          bg="#2563eb", fg="white", activebackground="#1e4fd0",
          relief="flat", padx=18, pady=8, command=analyse).pack(pady=4)
result_var = tk.StringVar()
result_label = tk.Label(root, textvariable=result_var, font=("Segoe UI", 14, "bold"), bg="white")
result_label.pack(pady=(18, 6))
detail_var = tk.StringVar()
tk.Label(root, textvariable=detail_var, font=("Consolas", 10), fg="#333", bg="white", justify="left").pack(pady=(0, 10))
tk.Label(root, text="Research baseline  \u00b7  ~74% accuracy  \u00b7  reports a probability, not a guarantee",
         font=("Segoe UI", 8), fg="#888", bg="white").pack(side="bottom", pady=12)
root.mainloop()
'''

with open("gui.py", "w", encoding="utf-8") as f:
    f.write(gui_code)
print("gui.py written to this folder.")

gui.py written to this folder.


In [3]:
import sys
!"{sys.executable}" gui.py

In [4]:
# ===== EDIT YOUR SLANG LIST HERE =====
SLANG = [
    "lekker","sharp sharp","sharp","the plug","no cap","hey","proper",
    "bru","yoh","eish","giving","clean clean","boss","skoen ding","oyiywa","kak","waram","daarsei","sho","nxa","shuu","mooi","best of the best","yesis"
    # --- add new Namibian slang below, each "in quotes", with a comma ---

]
# =====================================

import re, numpy as np, pandas as pd, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v = SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t): return 0.0

df = pd.read_csv("namibian_reviews_final.csv"); df["text"] = df["text"].astype(str)
y = df["final_label"].astype(int).values; raw = df["text"].values

def clean_text(t):
    t = t.lower(); t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z0-9\s!]", " ", t); return re.sub(r"\s+", " ", t).strip()
def eng(texts):
    out = []
    for t in texts:
        low, cc = t.lower(), len(t)
        out.append([len(t.split()), cc, t.count("!"),
            (sum(1 for c in t if c.isupper())/cc if cc else 0),
            sum(low.count(s) for s in SLANG), sentiment(t)])
    return np.array(out, dtype=float)

clean = np.array([clean_text(t) for t in raw])

# honest re-check with the new list (5-fold CV, fake-class F1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42); f1s = []
for tr, te in skf.split(clean, y):
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=2000)
    Xtr, Xte = vec.fit_transform(clean[tr]), vec.transform(clean[te])
    sc = MinMaxScaler()
    Xtr = hstack([Xtr, csr_matrix(sc.fit_transform(eng(raw[tr])))])
    Xte = hstack([Xte, csr_matrix(sc.transform(eng(raw[te])))])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, y[tr])
    f1s.append(f1_score(y[te], clf.predict(Xte), zero_division=0))
print(f"Slang words now: {len(SLANG)}")
print(f"Fake-class F1 (LR + features, 5-fold): {np.mean(f1s):.3f}   (was 0.513)")

# retrain demo model on ALL data + save
vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=2000)
X = vec.fit_transform(clean)
sc = MinMaxScaler(); Xf = hstack([X, csr_matrix(sc.fit_transform(eng(raw)))])
model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xf, y)
joblib.dump(model, "demo_model.pkl"); joblib.dump(vec, "demo_vectorizer.pkl"); joblib.dump(sc, "demo_scaler.pkl")

# sync gui.py to use the exact same list
with open("gui.py", "r", encoding="utf-8") as fh: src = fh.read()
src = re.sub(r"SLANG = \[[^\]]*\]", "SLANG = " + repr(SLANG), src, count=1)
with open("gui.py", "w", encoding="utf-8") as fh: fh.write(src)
print("Saved model + synced gui.py.")

Slang words now: 24
Fake-class F1 (LR + features, 5-fold): 0.485   (was 0.513)
Saved model + synced gui.py.


In [1]:
gui_code = r'''
import re
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import hstack, csr_matrix

SLANG = ["lekker", "sharp sharp", "sharp", "the plug", "no cap", "hey",
         "proper", "bru", "yoh", "eish", "giving", "clean clean", "boss"]

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _vader = SentimentIntensityAnalyzer()
    def sentiment(t):
        return _vader.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t):
        return 0.0

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z0-9\s!]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def eng_features(text):
    low, cc = text.lower(), len(text)
    return np.array([[
        len(text.split()), cc, text.count("!"),
        (sum(1 for c in text if c.isupper()) / cc if cc else 0),
        sum(low.count(s) for s in SLANG), sentiment(text),
    ]], dtype=float)

try:
    model = joblib.load("demo_model.pkl")
    vec = joblib.load("demo_vectorizer.pkl")
    scaler = joblib.load("demo_scaler.pkl")
    MODEL_OK = True
except Exception as e:
    MODEL_OK = False
    MODEL_ERR = str(e)

WHITE = "#ffffff"
root = tk.Tk()
root.title("Fake Review Detector - Namibian Online Consumer Reviews")
root.geometry("780x640")
root.configure(bg=WHITE)

tk.Label(root, text="Fake Review Detector", font=("Segoe UI", 17, "bold"),
         bg=WHITE).pack(pady=(14, 0))
tk.Label(root, text="A machine learning system for Namibian online consumer reviews",
         font=("Segoe UI", 10), fg="#555", bg=WHITE).pack(pady=(0, 10))

nb = ttk.Notebook(root)
nb.pack(fill="both", expand=True, padx=12, pady=8)

tab1 = tk.Frame(nb, bg=WHITE)
nb.add(tab1, text="  Detector  ")
tk.Label(tab1, text="Paste a review and classify it:", font=("Segoe UI", 11),
         bg=WHITE).pack(anchor="w", padx=20, pady=(14, 4))
input_box = scrolledtext.ScrolledText(tab1, height=5, width=78,
                                      font=("Segoe UI", 11), wrap="word")
input_box.pack(padx=20)
res_var = tk.StringVar()
det_var = tk.StringVar()

def analyse():
    if not MODEL_OK:
        messagebox.showerror("Model not loaded", "Could not load the model files:\n" + MODEL_ERR)
        return
    text = input_box.get("1.0", "end").strip()
    if not text:
        res_var.set("Please paste a review first."); det_var.set(""); return
    xf = hstack([vec.transform([clean_text(text)]),
                 csr_matrix(scaler.transform(eng_features(text)))])
    prob = float(model.predict_proba(xf)[0][1])
    if prob >= 0.5:
        res_var.set(f"\u26a0  LIKELY FAKE    (fake probability {prob:.0%})")
        res_lbl.config(fg="#b00020")
    else:
        res_var.set(f"\u2714  LIKELY GENUINE    (fake probability {prob:.0%})")
        res_lbl.config(fg="#1b7a1b")
    fe = eng_features(text)[0]
    det_var.set(f"words: {int(fe[0])}      characters: {int(fe[1])}      "
                f"exclamations: {int(fe[2])}\ncapitals ratio: {fe[3]:.2f}      "
                f"slang words: {int(fe[4])}      sentiment: {fe[5]:.2f}")

tk.Button(tab1, text="Analyse review", font=("Segoe UI", 12, "bold"),
          bg="#2563eb", fg="white", activebackground="#1e4fd0",
          relief="flat", padx=16, pady=7, command=analyse).pack(pady=12)
res_lbl = tk.Label(tab1, textvariable=res_var, font=("Segoe UI", 14, "bold"), bg=WHITE)
res_lbl.pack(pady=(6, 4))
tk.Label(tab1, textvariable=det_var, font=("Consolas", 10), fg="#333",
         bg=WHITE, justify="left").pack()
tk.Label(tab1, text="Research baseline  \u00b7  reports a probability, not a guarantee",
         font=("Segoe UI", 8), fg="#888", bg=WHITE).pack(side="bottom", pady=8)

tab2 = tk.Frame(nb, bg=WHITE)
nb.add(tab2, text="  Dataset  ")
try:
    df = pd.read_csv("namibian_reviews_final.csv")
    total = len(df)
    fl = pd.to_numeric(df["final_label"], errors="coerce")
    genuine = int((fl == 0).sum()); fake = int((fl == 1).sum())
    lines = [f"Total reviews collected and labelled:  {total}",
             f"Genuine (0):  {genuine}        Fake (1):  {fake}", ""]
    if "product_category" in df.columns:
        lines.append("By business category:")
        for cat, n in df["product_category"].value_counts().items():
            lines.append(f"    {cat}:  {n}")
        lines.append("")
    if "source" in df.columns:
        lines.append("By source:")
        for src, n in df["source"].value_counts().items():
            lines.append(f"    {src}:  {n}")
        lines.append("")
    if "label_annotator1" in df.columns and "label_annotator2" in df.columns:
        try:
            from sklearn.metrics import cohen_kappa_score
            a1 = pd.to_numeric(df["label_annotator1"], errors="coerce")
            a2 = pd.to_numeric(df["label_annotator2"], errors="coerce")
            m = a1.notna() & a2.notna()
            k = cohen_kappa_score(a1[m].astype(int), a2[m].astype(int))
            agree = int((a1[m].astype(int) == a2[m].astype(int)).sum())
            lines.append("Inter-annotator agreement (two independent labellers):")
            lines.append(f"    Agreed on {agree} of {int(m.sum())} reviews")
            lines.append(f"    Cohen's Kappa = {k:.3f}  (moderate agreement)")
        except Exception:
            pass
    tk.Label(tab2, text="\n".join(lines), font=("Consolas", 11),
             bg=WHITE, justify="left").pack(anchor="w", padx=22, pady=18)
except Exception as e:
    tk.Label(tab2, text="Could not load namibian_reviews_final.csv\n" + str(e),
             font=("Segoe UI", 10), fg="#b00020", bg=WHITE, justify="left").pack(anchor="w", padx=22, pady=18)

tab3 = tk.Frame(nb, bg=WHITE)
nb.add(tab3, text="  Model Results  ")
try:
    dfr = pd.read_csv("results_model_comparison.csv")
    tk.Label(tab3, text="Five-fold cross-validated performance (fake class)",
             font=("Segoe UI", 11, "bold"), bg=WHITE).pack(anchor="w", padx=20, pady=(16, 6))
    cols = list(dfr.columns)
    tv = ttk.Treeview(tab3, columns=cols, show="headings", height=min(12, len(dfr)))
    for c in cols:
        tv.heading(c, text=c); tv.column(c, width=130, anchor="center")
    for _, r in dfr.iterrows():
        tv.insert("", "end", values=[r[c] for c in cols])
    tv.pack(fill="x", padx=20)
    tk.Label(tab3, text="Best model: Logistic Regression  \u00b7  highest F1 on the fake class",
             font=("Segoe UI", 9), fg="#555", bg=WHITE).pack(anchor="w", padx=20, pady=8)
except Exception as e:
    tk.Label(tab3, text="Could not load results_model_comparison.csv\n" + str(e),
             font=("Segoe UI", 10), fg="#b00020", bg=WHITE, justify="left").pack(anchor="w", padx=22, pady=18)

tab4 = tk.Frame(nb, bg=WHITE)
nb.add(tab4, text="  Feature Importance  ")
try:
    dff = pd.read_csv("results_feature_importance.csv")
    tk.Label(tab4, text="What the model learned  (red = points to FAKE, green = points to GENUINE)",
             font=("Segoe UI", 11, "bold"), bg=WHITE).pack(anchor="w", padx=20, pady=(16, 10))
    cv = tk.Canvas(tab4, bg=WHITE, highlightthickness=0, height=320)
    cv.pack(fill="both", expand=True, padx=20)
    coefs = pd.to_numeric(dff["coefficient"], errors="coerce").fillna(0)
    maxabs = max(abs(coefs.min()), abs(coefs.max()), 0.001)
    midx = 360; y = 20
    for _, r in dff.iterrows():
        name = str(r["feature"]); c = float(r["coefficient"])
        length = int(abs(c) / maxabs * 150)
        cv.create_text(midx - 10, y + 10, text=name, anchor="e", font=("Segoe UI", 10))
        if c >= 0:
            cv.create_rectangle(midx, y, midx + length, y + 20, fill="#d65a5a", outline="")
            cv.create_text(midx + length + 6, y + 10, text=f"+{c:.2f}", anchor="w", font=("Consolas", 9))
        else:
            cv.create_rectangle(midx - length, y, midx, y + 20, fill="#5aa15a", outline="")
            cv.create_text(midx - length - 6, y + 10, text=f"{c:.2f}", anchor="e", font=("Consolas", 9))
        y += 34
    cv.create_line(midx, 10, midx, y, fill="#ccc")
except Exception as e:
    tk.Label(tab4, text="Could not load results_feature_importance.csv\n" + str(e),
             font=("Segoe UI", 10), fg="#b00020", bg=WHITE, justify="left").pack(anchor="w", padx=22, pady=18)

root.mainloop()
'''

with open("gui.py", "w", encoding="utf-8") as f:
    f.write(gui_code)
print("gui.py updated to the 4-tab version.")

gui.py updated to the 4-tab version.


In [3]:
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.sparse import hstack, csr_matrix

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v = SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t): return 0.0

df = pd.read_csv("namibian_reviews_final.csv")
df["text"] = df["text"].astype(str)
y = df["final_label"].astype(int).values
raw = df["text"].values

SLANG = ["lekker","sharp sharp","sharp","the plug","no cap","hey","proper",
         "bru","yoh","eish","giving","clean clean","boss"]
FEATNAMES = ["word_count","char_count","exclamations","caps_ratio","slang_count","sentiment"]

def clean_text(t):
    t=t.lower(); t=re.sub(r"http\S+"," ",t)
    t=re.sub(r"[^a-z0-9\s!]"," ",t); return re.sub(r"\s+"," ",t).strip()

def eng(texts):
    out=[]
    for t in texts:
        low,cc=t.lower(),len(t)
        out.append([len(t.split()),cc,t.count("!"),
            (sum(1 for c in t if c.isupper())/cc if cc else 0),
            sum(low.count(s) for s in SLANG), sentiment(t)])
    return np.array(out,dtype=float)

clean = np.array([clean_text(t) for t in raw])
models = {
    "Logistic Regression": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest":       lambda: RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "Naive Bayes":         lambda: MultinomialNB(),
}

def run_cv(use_features):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    res = {m: {"acc":[],"prec":[],"rec":[],"f1":[]} for m in models}
    for tr, te in skf.split(clean, y):
        vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=2000)
        Xtr, Xte = vec.fit_transform(clean[tr]), vec.transform(clean[te])
        if use_features:
            sc = MinMaxScaler()
            Xtr = hstack([Xtr, csr_matrix(sc.fit_transform(eng(raw[tr])))])
            Xte = hstack([Xte, csr_matrix(sc.transform(eng(raw[te])))])
        for name, make in models.items():
            clf = make(); clf.fit(Xtr, y[tr]); p = clf.predict(Xte)
            res[name]["acc"].append(accuracy_score(y[te], p))
            res[name]["prec"].append(precision_score(y[te], p, zero_division=0))
            res[name]["rec"].append(recall_score(y[te], p, zero_division=0))
            res[name]["f1"].append(f1_score(y[te], p, zero_division=0))
    return res

rows=[]
for flag,label in [(False,"TF-IDF only"),(True,"TF-IDF + features")]:
    r=run_cv(flag)
    for name,m in r.items():
        rows.append({"feature_set":label,"model":name,
                     "accuracy":round(np.mean(m["acc"]),3),
                     "precision":round(np.mean(m["prec"]),3),
                     "recall":round(np.mean(m["rec"]),3),
                     "f1":round(np.mean(m["f1"]),3)})
pd.DataFrame(rows).to_csv("results_model_comparison.csv", index=False)

vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=2000)
Xall = vec.fit_transform(clean)
sc = MinMaxScaler(); Xall_f = hstack([Xall, csr_matrix(sc.fit_transform(eng(raw)))])
lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xall_f, y)
pd.DataFrame({"feature": FEATNAMES, "coefficient": np.round(lr.coef_[0][-6:],3)}
            ).sort_values("coefficient", ascending=False).to_csv("results_feature_importance.csv", index=False)

print("Saved -> results_model_comparison.csv, results_feature_importance.csv")

Saved -> results_model_comparison.csv, results_feature_importance.csv


In [4]:
import sys
!"{sys.executable}" gui.py

In [5]:
gui_code = r'''
import re
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import hstack, csr_matrix

SLANG = ["lekker", "sharp sharp", "sharp", "the plug", "no cap", "hey",
         "proper", "bru", "yoh", "eish", "giving", "clean clean", "boss"]

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _vader = SentimentIntensityAnalyzer()
    def sentiment(t):
        return _vader.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t):
        return 0.0

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z0-9\s!]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def eng_features(text):
    low, cc = text.lower(), len(text)
    return np.array([[
        len(text.split()), cc, text.count("!"),
        (sum(1 for c in text if c.isupper()) / cc if cc else 0),
        sum(low.count(s) for s in SLANG), sentiment(text),
    ]], dtype=float)

try:
    model = joblib.load("demo_model.pkl")
    vec = joblib.load("demo_vectorizer.pkl")
    scaler = joblib.load("demo_scaler.pkl")
    MODEL_OK = True
except Exception as e:
    MODEL_OK = False
    MODEL_ERR = str(e)

WHITE = "#ffffff"
root = tk.Tk()
root.title("Fake Review Detector - Namibian Online Consumer Reviews")
root.geometry("820x700")
root.configure(bg=WHITE)

header = tk.Frame(root, bg=WHITE)
header.pack(fill="x", pady=(16, 6))
tk.Label(header, text="Fake Review Detector", font=("Segoe UI", 18, "bold"),
         bg=WHITE).pack()
tk.Label(header, text="A machine learning system for Namibian online consumer reviews",
         font=("Segoe UI", 10), fg="#555", bg=WHITE).pack()

nb = ttk.Notebook(root)
nb.pack(fill="both", expand=True, padx=14, pady=10)

# ---------- TAB 1: DETECTOR ----------
tab1 = tk.Frame(nb, bg=WHITE)
nb.add(tab1, text="  Detector  ")
body1 = tk.Frame(tab1, bg=WHITE)
body1.pack(fill="both", expand=True, padx=24, pady=18)

tk.Label(body1, text="Paste a review and classify it:", font=("Segoe UI", 11),
         bg=WHITE).pack(anchor="w", pady=(0, 6))
input_box = scrolledtext.ScrolledText(body1, height=5, font=("Segoe UI", 11), wrap="word")
input_box.pack(fill="x")

res_var = tk.StringVar()
det_var = tk.StringVar()

def analyse():
    if not MODEL_OK:
        messagebox.showerror("Model not loaded", "Could not load the model files:\n" + MODEL_ERR)
        return
    text = input_box.get("1.0", "end").strip()
    if not text:
        res_var.set("Please paste a review first."); det_var.set(""); return
    xf = hstack([vec.transform([clean_text(text)]),
                 csr_matrix(scaler.transform(eng_features(text)))])
    prob = float(model.predict_proba(xf)[0][1])
    if prob >= 0.5:
        res_var.set(f"\u26a0  LIKELY FAKE    (fake probability {prob:.0%})")
        res_lbl.config(fg="#b00020")
    else:
        res_var.set(f"\u2714  LIKELY GENUINE    (fake probability {prob:.0%})")
        res_lbl.config(fg="#1b7a1b")
    fe = eng_features(text)[0]
    det_var.set(f"words: {int(fe[0])}      characters: {int(fe[1])}      "
                f"exclamations: {int(fe[2])}      capitals ratio: {fe[3]:.2f}      "
                f"slang words: {int(fe[4])}      sentiment: {fe[5]:.2f}")

tk.Button(body1, text="Analyse review", font=("Segoe UI", 12, "bold"),
          bg="#2563eb", fg="white", activebackground="#1e4fd0",
          relief="flat", padx=16, pady=7, command=analyse).pack(pady=16)
res_lbl = tk.Label(body1, textvariable=res_var, font=("Segoe UI", 14, "bold"), bg=WHITE)
res_lbl.pack(pady=(4, 8))
tk.Label(body1, textvariable=det_var, font=("Consolas", 10), fg="#333",
         bg=WHITE, justify="left", wraplength=720).pack()
tk.Label(body1, text="Research baseline  \u00b7  reports a probability, not a guarantee",
         font=("Segoe UI", 8), fg="#888", bg=WHITE).pack(pady=(20, 0))

# ---------- TAB 2: DATASET ----------
tab2 = tk.Frame(nb, bg=WHITE)
nb.add(tab2, text="  Dataset  ")
try:
    df = pd.read_csv("namibian_reviews_final.csv")
    total = len(df)
    fl = pd.to_numeric(df["final_label"], errors="coerce")
    genuine = int((fl == 0).sum()); fake = int((fl == 1).sum())
    lines = [f"Total reviews collected and labelled:  {total}",
             f"Genuine (0):  {genuine}        Fake (1):  {fake}", ""]
    if "product_category" in df.columns:
        lines.append("By business category:")
        for cat, n in df["product_category"].value_counts().items():
            lines.append(f"    {cat}:  {n}")
        lines.append("")
    if "source" in df.columns:
        lines.append("By source:")
        for src, n in df["source"].value_counts().items():
            lines.append(f"    {src}:  {n}")
        lines.append("")
    if "label_annotator1" in df.columns and "label_annotator2" in df.columns:
        try:
            from sklearn.metrics import cohen_kappa_score
            a1 = pd.to_numeric(df["label_annotator1"], errors="coerce")
            a2 = pd.to_numeric(df["label_annotator2"], errors="coerce")
            m = a1.notna() & a2.notna()
            k = cohen_kappa_score(a1[m].astype(int), a2[m].astype(int))
            agree = int((a1[m].astype(int) == a2[m].astype(int)).sum())
            lines.append("Inter-annotator agreement (two independent labellers):")
            lines.append(f"    Agreed on {agree} of {int(m.sum())} reviews")
            lines.append(f"    Cohen's Kappa = {k:.3f}  (moderate agreement)")
        except Exception:
            pass
    tk.Label(tab2, text="\n".join(lines), font=("Consolas", 11),
             bg=WHITE, justify="left").pack(anchor="w", padx=26, pady=20)
except Exception as e:
    tk.Label(tab2, text="Could not load namibian_reviews_final.csv\n" + str(e),
             font=("Segoe UI", 10), fg="#b00020", bg=WHITE, justify="left").pack(anchor="w", padx=26, pady=20)

# ---------- TAB 3: MODEL RESULTS ----------
tab3 = tk.Frame(nb, bg=WHITE)
nb.add(tab3, text="  Model Results  ")
try:
    dfr = pd.read_csv("results_model_comparison.csv")
    tk.Label(tab3, text="Five-fold cross-validated performance (fake class)",
             font=("Segoe UI", 11, "bold"), bg=WHITE).pack(anchor="w", padx=24, pady=(20, 8))
    cols = list(dfr.columns)
    tv = ttk.Treeview(tab3, columns=cols, show="headings", height=min(12, len(dfr)))
    for c in cols:
        tv.heading(c, text=c); tv.column(c, width=140, anchor="center")
    for _, r in dfr.iterrows():
        tv.insert("", "end", values=[r[c] for c in cols])
    tv.pack(fill="x", padx=24)
    tk.Label(tab3, text="Best model: Logistic Regression  \u00b7  highest F1 on the fake class",
             font=("Segoe UI", 9), fg="#555", bg=WHITE).pack(anchor="w", padx=24, pady=10)
except Exception as e:
    tk.Label(tab3, text="Could not load results_model_comparison.csv\n" + str(e),
             font=("Segoe UI", 10), fg="#b00020", bg=WHITE, justify="left").pack(anchor="w", padx=26, pady=20)

# ---------- TAB 4: FEATURE IMPORTANCE ----------
tab4 = tk.Frame(nb, bg=WHITE)
nb.add(tab4, text="  Feature Importance  ")
try:
    dff = pd.read_csv("results_feature_importance.csv")
    tk.Label(tab4, text="What the model learned  (red = points to FAKE, green = points to GENUINE)",
             font=("Segoe UI", 11, "bold"), bg=WHITE).pack(anchor="w", padx=24, pady=(20, 12))
    cv = tk.Canvas(tab4, bg=WHITE, highlightthickness=0, height=330)
    cv.pack(fill="both", expand=True, padx=24)
    coefs = pd.to_numeric(dff["coefficient"], errors="coerce").fillna(0)
    maxabs = max(abs(coefs.min()), abs(coefs.max()), 0.001)
    midx = 380; y = 20
    for _, r in dff.iterrows():
        name = str(r["feature"]); c = float(r["coefficient"])
        length = int(abs(c) / maxabs * 150)
        cv.create_text(midx - 12, y + 10, text=name, anchor="e", font=("Segoe UI", 10))
        if c >= 0:
            cv.create_rectangle(midx, y, midx + length, y + 20, fill="#d65a5a", outline="")
            cv.create_text(midx + length + 6, y + 10, text=f"+{c:.2f}", anchor="w", font=("Consolas", 9))
        else:
            cv.create_rectangle(midx - length, y, midx, y + 20, fill="#5aa15a", outline="")
            cv.create_text(midx - length - 6, y + 10, text=f"{c:.2f}", anchor="e", font=("Consolas", 9))
        y += 36
    cv.create_line(midx, 10, midx, y, fill="#ccc")
except Exception as e:
    tk.Label(tab4, text="Could not load results_feature_importance.csv\n" + str(e),
             font=("Segoe UI", 10), fg="#b00020", bg=WHITE, justify="left").pack(anchor="w", padx=26, pady=20)

root.mainloop()
'''

with open("gui.py", "w", encoding="utf-8") as f:
    f.write(gui_code)
print("gui.py updated (layout fixed).")

gui.py updated (layout fixed).


In [6]:
import sys
!"{sys.executable}" gui.py

In [7]:
with open("gui.py", "r", encoding="utf-8") as f:
    src = f.read()

old_start = '    coefs = pd.to_numeric(dff["coefficient"]'
old_end = 'cv.create_line(midx, 10, midx, y, fill="#ccc")'
i = src.index(old_start)
j = src.index(old_end) + len(old_end)

new_block = '''    coefs = pd.to_numeric(dff["coefficient"], errors="coerce").fillna(0)
    maxabs = max(abs(coefs.min()), abs(coefs.max()), 0.001)
    base = 180; y = 20
    for _, r in dff.iterrows():
        name = str(r["feature"]); c = float(r["coefficient"])
        length = int(abs(c) / maxabs * 200)
        colour = "#d65a5a" if c >= 0 else "#5aa15a"
        points = "FAKE" if c >= 0 else "GENUINE"
        cv.create_text(20, y + 10, text=name, anchor="w", font=("Segoe UI", 10))
        cv.create_rectangle(base, y, base + length, y + 20, fill=colour, outline="")
        cv.create_text(base + length + 8, y + 10, text=f"{c:+.2f}  ({points})", anchor="w", font=("Consolas", 9))
        y += 38'''

src = src[:i] + new_block + src[j:]
with open("gui.py", "w", encoding="utf-8") as f:
    f.write(src)
print("Feature Importance tab layout fixed.")

Feature Importance tab layout fixed.


In [8]:
import sys
!"{sys.executable}" gui.py

In [9]:
gui_code = r'''
import re
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import hstack, csr_matrix

SIDEBAR = "#0f172a"
SIDEBAR_ACT = "#1e293b"
ACCENT = "#2563eb"
BG = "#f1f5f9"
CARD = "#ffffff"
LINE = "#e2e8f0"
TEXT = "#0f172a"
MUTED = "#64748b"
GREEN = "#15803d"; GREEN_BG = "#dcfce7"
RED = "#b91c1c"; RED_BG = "#fee2e2"

SLANG = ["lekker", "sharp sharp", "sharp", "the plug", "no cap", "hey",
         "proper", "bru", "yoh", "eish", "giving", "clean clean", "boss"]

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _vader = SentimentIntensityAnalyzer()
    def sentiment(t):
        return _vader.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t):
        return 0.0

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z0-9\s!]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def eng_features(text):
    low, cc = text.lower(), len(text)
    return np.array([[
        len(text.split()), cc, text.count("!"),
        (sum(1 for c in text if c.isupper()) / cc if cc else 0),
        sum(low.count(s) for s in SLANG), sentiment(text),
    ]], dtype=float)

try:
    model = joblib.load("demo_model.pkl")
    vec = joblib.load("demo_vectorizer.pkl")
    scaler = joblib.load("demo_scaler.pkl")
    MODEL_OK = True
except Exception as e:
    MODEL_OK = False
    MODEL_ERR = str(e)

root = tk.Tk()
root.title("Fake Review Detector")
root.geometry("1000x660")
root.configure(bg=BG)
root.minsize(900, 600)

style = ttk.Style()
try:
    style.theme_use("clam")
except Exception:
    pass
style.configure("Modern.Treeview", background=CARD, fieldbackground=CARD,
                foreground=TEXT, rowheight=30, font=("Segoe UI", 10), borderwidth=0)
style.configure("Modern.Treeview.Heading", background="#e2e8f0", foreground=TEXT,
                font=("Segoe UI", 10, "bold"), relief="flat", padding=6)
style.map("Modern.Treeview", background=[("selected", "#dbeafe")],
          foreground=[("selected", TEXT)])

root.grid_columnconfigure(1, weight=1)
root.grid_rowconfigure(0, weight=1)

side = tk.Frame(root, bg=SIDEBAR, width=220)
side.grid(row=0, column=0, sticky="nsew")
side.grid_propagate(False)
tk.Label(side, text="  Fake Review\n  Detector", font=("Segoe UI", 15, "bold"),
         fg="white", bg=SIDEBAR, justify="left").pack(anchor="w", pady=(24, 2), padx=6)
tk.Label(side, text="  Namibian consumer reviews", font=("Segoe UI", 8),
         fg="#94a3b8", bg=SIDEBAR).pack(anchor="w", padx=6, pady=(0, 24))

navbtns = {}
def show(key):
    pages[key].tkraise()
    for k, b in navbtns.items():
        if k == key:
            b.config(bg=SIDEBAR_ACT, fg="white")
        else:
            b.config(bg=SIDEBAR, fg="#cbd5e1")

def make_nav(label, key):
    b = tk.Button(side, text="   " + label, font=("Segoe UI", 11), fg="#cbd5e1",
                  bg=SIDEBAR, activebackground=SIDEBAR_ACT, activeforeground="white",
                  relief="flat", anchor="w", padx=16, pady=12, bd=0,
                  command=lambda: show(key))
    b.pack(fill="x")
    navbtns[key] = b

make_nav("Detector", "detector")
make_nav("Dataset", "dataset")
make_nav("Model Results", "results")
make_nav("Feature Importance", "features")
tk.Label(side, text="  Honours Research\n  Prototype", font=("Segoe UI", 8),
         fg="#64748b", bg=SIDEBAR, justify="left").pack(side="bottom", anchor="w", padx=6, pady=16)

content = tk.Frame(root, bg=BG)
content.grid(row=0, column=1, sticky="nsew")
content.grid_rowconfigure(0, weight=1)
content.grid_columnconfigure(0, weight=1)

pages = {}
def new_page():
    p = tk.Frame(content, bg=BG)
    p.grid(row=0, column=0, sticky="nsew")
    return p

def card(parent, **kw):
    return tk.Frame(parent, bg=CARD, highlightbackground=LINE, highlightthickness=1, **kw)

def stat_card(parent, value, label, accent=ACCENT):
    c = card(parent)
    tk.Label(c, text=value, font=("Segoe UI", 22, "bold"), fg=accent, bg=CARD).pack(anchor="w", padx=16, pady=(14, 0))
    tk.Label(c, text=label, font=("Segoe UI", 9), fg=MUTED, bg=CARD).pack(anchor="w", padx=16, pady=(0, 14))
    return c

def page_title(parent, title, sub):
    tk.Label(parent, text=title, font=("Segoe UI", 18, "bold"), fg=TEXT, bg=BG).pack(anchor="w", padx=28, pady=(24, 0))
    tk.Label(parent, text=sub, font=("Segoe UI", 10), fg=MUTED, bg=BG).pack(anchor="w", padx=28, pady=(0, 16))

pd_ = new_page(); pages["detector"] = pd_
page_title(pd_, "Review Detector", "Paste a review and the model estimates whether it is genuine or fake.")
dc = card(pd_); dc.pack(fill="both", expand=True, padx=28, pady=(0, 24))
tk.Label(dc, text="Review text", font=("Segoe UI", 10, "bold"), fg=TEXT, bg=CARD).pack(anchor="w", padx=18, pady=(16, 6))
input_box = scrolledtext.ScrolledText(dc, height=5, font=("Segoe UI", 11), wrap="word", relief="solid", borderwidth=1)
input_box.pack(fill="x", padx=18)
verdict_fr = tk.Frame(dc, bg=CARD)
verdict_fr.pack(fill="x", padx=18, pady=16)
verdict_lbl = tk.Label(verdict_fr, text="", font=("Segoe UI", 14, "bold"), bg=CARD, anchor="w", padx=16, pady=14)
feat_lbl = tk.Label(dc, text="", font=("Consolas", 10), fg="#334155", bg=CARD, justify="left")
feat_lbl.pack(anchor="w", padx=18)

def analyse():
    if not MODEL_OK:
        messagebox.showerror("Model not loaded", "Could not load model files:\n" + MODEL_ERR); return
    text = input_box.get("1.0", "end").strip()
    if not text:
        return
    xf = hstack([vec.transform([clean_text(text)]), csr_matrix(scaler.transform(eng_features(text)))])
    prob = float(model.predict_proba(xf)[0][1])
    verdict_lbl.pack(fill="x")
    if prob >= 0.5:
        verdict_fr.config(bg=RED_BG)
        verdict_lbl.config(text=f"\u26a0  LIKELY FAKE        fake probability  {prob:.0%}", bg=RED_BG, fg=RED)
    else:
        verdict_fr.config(bg=GREEN_BG)
        verdict_lbl.config(text=f"\u2714  LIKELY GENUINE        fake probability  {prob:.0%}", bg=GREEN_BG, fg=GREEN)
    fe = eng_features(text)[0]
    feat_lbl.config(text=(f"words {int(fe[0])}    characters {int(fe[1])}    exclamations {int(fe[2])}    "
                          f"capitals {fe[3]:.2f}    slang {int(fe[4])}    sentiment {fe[5]:.2f}"))

tk.Button(dc, text="Analyse review", font=("Segoe UI", 11, "bold"), fg="white", bg=ACCENT,
          activebackground="#1e4fd0", activeforeground="white", relief="flat", padx=20, pady=9,
          cursor="hand2", command=analyse).pack(anchor="w", padx=18, pady=(14, 18))

pg = new_page(); pages["dataset"] = pg
page_title(pg, "Dataset", "The labelled Namibian review dataset built for this study.")
try:
    df = pd.read_csv("namibian_reviews_final.csv")
    total = len(df)
    fl = pd.to_numeric(df["final_label"], errors="coerce")
    genuine = int((fl == 0).sum()); fake = int((fl == 1).sum())
    kappa_txt = "n/a"
    if "label_annotator1" in df.columns and "label_annotator2" in df.columns:
        try:
            from sklearn.metrics import cohen_kappa_score
            a1 = pd.to_numeric(df["label_annotator1"], errors="coerce")
            a2 = pd.to_numeric(df["label_annotator2"], errors="coerce")
            m = a1.notna() & a2.notna()
            kappa_txt = f"{cohen_kappa_score(a1[m].astype(int), a2[m].astype(int)):.3f}"
        except Exception:
            pass
    row = tk.Frame(pg, bg=BG); row.pack(fill="x", padx=22)
    for i, (v, l, col) in enumerate([(str(total), "Total reviews", ACCENT), (str(genuine), "Genuine", GREEN),
                                     (str(fake), "Fake", RED), (kappa_txt, "Cohen's Kappa", "#7c3aed")]):
        sc = stat_card(row, v, l, col); sc.grid(row=0, column=i, padx=6, sticky="ew")
        row.grid_columnconfigure(i, weight=1)
    cols = tk.Frame(pg, bg=BG); cols.pack(fill="both", expand=True, padx=22, pady=16)
    cols.grid_columnconfigure(0, weight=1); cols.grid_columnconfigure(1, weight=1)
    if "product_category" in df.columns:
        c1 = card(cols); c1.grid(row=0, column=0, padx=6, sticky="nsew")
        tk.Label(c1, text="By business category", font=("Segoe UI", 11, "bold"), fg=TEXT, bg=CARD).pack(anchor="w", padx=16, pady=(14, 6))
        for cat, n in df["product_category"].value_counts().items():
            tk.Label(c1, text=f"{cat}:  {n}", font=("Segoe UI", 10), fg="#334155", bg=CARD).pack(anchor="w", padx=16, pady=1)
        tk.Label(c1, text="", bg=CARD).pack(pady=4)
    if "source" in df.columns:
        c2 = card(cols); c2.grid(row=0, column=1, padx=6, sticky="nsew")
        tk.Label(c2, text="By source", font=("Segoe UI", 11, "bold"), fg=TEXT, bg=CARD).pack(anchor="w", padx=16, pady=(14, 6))
        for src, n in df["source"].value_counts().items():
            tk.Label(c2, text=f"{src}:  {n}", font=("Segoe UI", 10), fg="#334155", bg=CARD).pack(anchor="w", padx=16, pady=1)
        tk.Label(c2, text="Two annotators labelled independently;\nKappa measures their agreement.",
                 font=("Segoe UI", 9), fg=MUTED, bg=CARD, justify="left").pack(anchor="w", padx=16, pady=(10, 14))
except Exception as e:
    tk.Label(pg, text="Could not load namibian_reviews_final.csv\n" + str(e), font=("Segoe UI", 10), fg=RED, bg=BG, justify="left").pack(anchor="w", padx=28)

pr = new_page(); pages["results"] = pr
page_title(pr, "Model Results", "Five-fold cross-validated performance on the fake class.")
try:
    dfr = pd.read_csv("results_model_comparison.csv")
    tc = card(pr); tc.pack(fill="x", padx=28, pady=(0, 10))
    cols = list(dfr.columns)
    tv = ttk.Treeview(tc, columns=cols, show="headings", height=len(dfr), style="Modern.Treeview")
    for c in cols:
        tv.heading(c, text=c.replace("_", " ").title()); tv.column(c, width=150, anchor="center")
    for n, (_, r) in enumerate(dfr.iterrows()):
        tv.insert("", "end", values=[r[c] for c in cols], tags=("odd" if n % 2 else "even",))
    tv.tag_configure("odd", background="#f8fafc"); tv.tag_configure("even", background=CARD)
    tv.pack(fill="x", padx=10, pady=10)
    cal = tk.Frame(pr, bg="#eff6ff", highlightbackground="#bfdbfe", highlightthickness=1)
    cal.pack(fill="x", padx=28, pady=6)
    tk.Label(cal, text="Best model:  Logistic Regression  \u2014  highest F1 on the fake class",
             font=("Segoe UI", 10, "bold"), fg="#1e40af", bg="#eff6ff").pack(anchor="w", padx=14, pady=10)
except Exception as e:
    tk.Label(pr, text="Could not load results_model_comparison.csv\n" + str(e), font=("Segoe UI", 10), fg=RED, bg=BG, justify="left").pack(anchor="w", padx=28)

pf = new_page(); pages["features"] = pf
page_title(pf, "Feature Importance", "What the model learned  \u2014  red points to fake, green points to genuine.")
try:
    dff = pd.read_csv("results_feature_importance.csv")
    fc = card(pf); fc.pack(fill="both", expand=True, padx=28, pady=(0, 24))
    cv = tk.Canvas(fc, bg=CARD, highlightthickness=0); cv.pack(fill="both", expand=True, padx=14, pady=14)
    coefs = pd.to_numeric(dff["coefficient"], errors="coerce").fillna(0)
    maxabs = max(abs(coefs.min()), abs(coefs.max()), 0.001)
    base = 190; y = 24
    for _, r in dff.iterrows():
        name = str(r["feature"]); c = float(r["coefficient"])
        length = int(abs(c) / maxabs * 230)
        colour = "#dc2626" if c >= 0 else "#16a34a"
        tag = "FAKE" if c >= 0 else "GENUINE"
        cv.create_text(24, y + 11, text=name, anchor="w", font=("Segoe UI", 11))
        cv.create_rectangle(base, y, base + length, y + 22, fill=colour, outline="")
        cv.create_text(base + length + 10, y + 11, text=f"{c:+.2f}  {tag}", anchor="w", font=("Consolas", 9), fill="#334155")
        y += 42
except Exception as e:
    tk.Label(pf, text="Could not load results_feature_importance.csv\n" + str(e), font=("Segoe UI", 10), fg=RED, bg=BG, justify="left").pack(anchor="w", padx=28)

show("detector")
root.mainloop()
'''

with open("gui.py", "w", encoding="utf-8") as f:
    f.write(gui_code)
print("gui.py updated to the modern version.")

gui.py updated to the modern version.


In [10]:
import sys
!"{sys.executable}" gui.py

In [1]:
gui_code = r'''
import re
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox, filedialog
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import hstack, csr_matrix

SIDEBAR = "#0f172a"; SIDEBAR_ACT = "#1e293b"; ACCENT = "#2563eb"
BG = "#f1f5f9"; CARD = "#ffffff"; LINE = "#e2e8f0"; TEXT = "#0f172a"; MUTED = "#64748b"
GREEN = "#15803d"; GREEN_BG = "#dcfce7"; RED = "#b91c1c"; RED_BG = "#fee2e2"

SLANG = ["lekker", "sharp sharp", "sharp", "the plug", "no cap", "hey",
         "proper", "bru", "yoh", "eish", "giving", "clean clean", "boss"]

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _vader = SentimentIntensityAnalyzer()
    def sentiment(t):
        return _vader.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t):
        return 0.0

def clean_text(t):
    t = t.lower(); t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"[^a-z0-9\s!]", " ", t); return re.sub(r"\s+", " ", t).strip()

def eng_features(text):
    low, cc = text.lower(), len(text)
    return np.array([[len(text.split()), cc, text.count("!"),
        (sum(1 for c in text if c.isupper()) / cc if cc else 0),
        sum(low.count(s) for s in SLANG), sentiment(text)]], dtype=float)

try:
    model = joblib.load("demo_model.pkl")
    vec = joblib.load("demo_vectorizer.pkl")
    scaler = joblib.load("demo_scaler.pkl")
    MODEL_OK = True
    try:
        FEAT_NAMES = list(vec.get_feature_names_out())
    except Exception:
        FEAT_NAMES = list(vec.get_feature_names())
    N_VOCAB = len(FEAT_NAMES)
except Exception as e:
    MODEL_OK = False; MODEL_ERR = str(e)

def classify_many(texts):
    clean = [clean_text(t) for t in texts]
    X = vec.transform(clean)
    F = scaler.transform(np.vstack([eng_features(t)[0] for t in texts]))
    return model.predict_proba(hstack([X, csr_matrix(F)]))[:, 1]

def top_words(text, k=8):
    row = vec.transform([clean_text(text)]).tocoo()
    coef = model.coef_[0]
    contribs = [(FEAT_NAMES[i], v * coef[i]) for i, v in zip(row.col, row.data) if i < N_VOCAB]
    contribs.sort(key=lambda x: abs(x[1]), reverse=True)
    return contribs[:k]

root = tk.Tk()
root.title("Fake Review Detector")
root.geometry("1040x680"); root.configure(bg=BG); root.minsize(940, 620)

style = ttk.Style()
try: style.theme_use("clam")
except Exception: pass
style.configure("Modern.Treeview", background=CARD, fieldbackground=CARD, foreground=TEXT,
                rowheight=28, font=("Segoe UI", 10), borderwidth=0)
style.configure("Modern.Treeview.Heading", background="#e2e8f0", foreground=TEXT,
                font=("Segoe UI", 10, "bold"), relief="flat", padding=6)
style.map("Modern.Treeview", background=[("selected", "#dbeafe")], foreground=[("selected", TEXT)])

root.grid_columnconfigure(1, weight=1); root.grid_rowconfigure(0, weight=1)

side = tk.Frame(root, bg=SIDEBAR, width=220); side.grid(row=0, column=0, sticky="nsew"); side.grid_propagate(False)
tk.Label(side, text="  Fake Review\n  Detector", font=("Segoe UI", 15, "bold"), fg="white",
         bg=SIDEBAR, justify="left").pack(anchor="w", pady=(24, 2), padx=6)
tk.Label(side, text="  Namibian consumer reviews", font=("Segoe UI", 8), fg="#94a3b8",
         bg=SIDEBAR).pack(anchor="w", padx=6, pady=(0, 24))

navbtns = {}
def show(key):
    pages[key].tkraise()
    for k, b in navbtns.items():
        b.config(bg=SIDEBAR_ACT if k == key else SIDEBAR, fg="white" if k == key else "#cbd5e1")

def make_nav(label, key):
    b = tk.Button(side, text="   " + label, font=("Segoe UI", 11), fg="#cbd5e1", bg=SIDEBAR,
                  activebackground=SIDEBAR_ACT, activeforeground="white", relief="flat",
                  anchor="w", padx=16, pady=12, bd=0, command=lambda: show(key))
    b.pack(fill="x"); navbtns[key] = b

for lab, key in [("Detector", "detector"), ("Batch Scan", "batch"), ("Dataset", "dataset"),
                 ("Model Results", "results"), ("Feature Importance", "features")]:
    make_nav(lab, key)

content = tk.Frame(root, bg=BG); content.grid(row=0, column=1, sticky="nsew")
content.grid_rowconfigure(0, weight=1); content.grid_columnconfigure(0, weight=1)
pages = {}
def new_page():
    p = tk.Frame(content, bg=BG); p.grid(row=0, column=0, sticky="nsew"); return p
def card(parent, **kw):
    return tk.Frame(parent, bg=CARD, highlightbackground=LINE, highlightthickness=1, **kw)
def stat_card(parent, value, label, accent=ACCENT):
    c = card(parent)
    tk.Label(c, text=value, font=("Segoe UI", 22, "bold"), fg=accent, bg=CARD).pack(anchor="w", padx=16, pady=(14, 0))
    tk.Label(c, text=label, font=("Segoe UI", 9), fg=MUTED, bg=CARD).pack(anchor="w", padx=16, pady=(0, 14))
    return c
def page_title(parent, title, sub):
    tk.Label(parent, text=title, font=("Segoe UI", 18, "bold"), fg=TEXT, bg=BG).pack(anchor="w", padx=28, pady=(22, 0))
    tk.Label(parent, text=sub, font=("Segoe UI", 10), fg=MUTED, bg=BG).pack(anchor="w", padx=28, pady=(0, 14))

p1 = new_page(); pages["detector"] = p1
page_title(p1, "Review Detector", "Classify a single review and see the evidence behind the verdict.")
dc = card(p1); dc.pack(fill="both", expand=True, padx=28, pady=(0, 22))
tk.Label(dc, text="Review text", font=("Segoe UI", 10, "bold"), fg=TEXT, bg=CARD).pack(anchor="w", padx=18, pady=(16, 6))
input_box = scrolledtext.ScrolledText(dc, height=4, font=("Segoe UI", 11), wrap="word", relief="solid", borderwidth=1)
input_box.pack(fill="x", padx=18)
exrow = tk.Frame(dc, bg=CARD); exrow.pack(anchor="w", padx=18, pady=(8, 0))
def fill_example(t):
    input_box.delete("1.0", "end"); input_box.insert("1.0", t)
tk.Label(exrow, text="Try:", font=("Segoe UI", 9), fg=MUTED, bg=CARD).pack(side="left")
for lbl, txt in [("hype example", "BEST SHOP EVER!!! amazing prices, highly recommend to everyone!!!"),
                 ("genuine example", "I bought a charger here last week, it works fine and they gave me a receipt.")]:
    tk.Button(exrow, text=lbl, font=("Segoe UI", 8), fg=ACCENT, bg="#eff6ff", relief="flat",
              padx=8, pady=2, cursor="hand2", command=lambda t=txt: fill_example(t)).pack(side="left", padx=4)
verdict_fr = tk.Frame(dc, bg=CARD); verdict_fr.pack(fill="x", padx=18, pady=(14, 6))
verdict_lbl = tk.Label(verdict_fr, text="", font=("Segoe UI", 14, "bold"), bg=CARD, anchor="w", padx=16, pady=12)
feat_lbl = tk.Label(dc, text="", font=("Consolas", 10), fg="#334155", bg=CARD, justify="left")
feat_lbl.pack(anchor="w", padx=18)
ev_title = tk.Label(dc, text="", font=("Segoe UI", 10, "bold"), fg=TEXT, bg=CARD)
ev_title.pack(anchor="w", padx=18, pady=(10, 2))
ev_fr = tk.Frame(dc, bg=CARD); ev_fr.pack(anchor="w", padx=16, pady=(0, 10), fill="x")

def analyse():
    if not MODEL_OK:
        messagebox.showerror("Model not loaded", "Could not load model files:\n" + MODEL_ERR); return
    text = input_box.get("1.0", "end").strip()
    if not text: return
    prob = float(classify_many([text])[0])
    verdict_lbl.pack(fill="x")
    if prob >= 0.5:
        verdict_fr.config(bg=RED_BG); verdict_lbl.config(text=f"\u26a0  LIKELY FAKE        fake probability  {prob:.0%}", bg=RED_BG, fg=RED)
    else:
        verdict_fr.config(bg=GREEN_BG); verdict_lbl.config(text=f"\u2714  LIKELY GENUINE        fake probability  {prob:.0%}", bg=GREEN_BG, fg=GREEN)
    fe = eng_features(text)[0]
    feat_lbl.config(text=(f"words {int(fe[0])}    characters {int(fe[1])}    exclamations {int(fe[2])}    "
                          f"capitals {fe[3]:.2f}    slang {int(fe[4])}    sentiment {fe[5]:.2f}"))
    for w in ev_fr.winfo_children(): w.destroy()
    words = top_words(text)
    if words:
        ev_title.config(text="Words that drove this verdict:")
        for i, (term, contrib) in enumerate(words):
            if contrib >= 0:
                bgc, fgc, mark = RED_BG, RED, "\u2191fake"
            else:
                bgc, fgc, mark = GREEN_BG, GREEN, "\u2193genuine"
            tk.Label(ev_fr, text=f" {term}  ({mark}) ", font=("Segoe UI", 9), bg=bgc, fg=fgc, padx=6, pady=3).grid(row=i // 4, column=i % 4, padx=4, pady=4, sticky="w")
    else:
        ev_title.config(text="No words from this review were in the model's vocabulary.")

tk.Button(dc, text="Analyse review", font=("Segoe UI", 11, "bold"), fg="white", bg=ACCENT,
          activebackground="#1e4fd0", activeforeground="white", relief="flat", padx=20, pady=9,
          cursor="hand2", command=analyse).pack(anchor="w", padx=18, pady=(12, 16))

p2 = new_page(); pages["batch"] = p2
page_title(p2, "Batch Scan", "Load a CSV of many reviews and classify them all at once.")
bbar = tk.Frame(p2, bg=BG); bbar.pack(fill="x", padx=28)
batch_info = tk.StringVar(value="No file loaded.")
batch_state = {"df": None}

def load_batch():
    path = filedialog.askopenfilename(title="Choose a CSV of reviews", filetypes=[("CSV files", "*.csv"), ("All files", "*.*")])
    if not path: return
    try:
        df = pd.read_csv(path)
    except Exception as e:
        messagebox.showerror("Could not read file", str(e)); return
    textcol = None
    for cand in ["text", "review", "Review", "Text", "content"]:
        if cand in df.columns: textcol = cand; break
    if textcol is None:
        obj = [c for c in df.columns if df[c].dtype == object]
        if not obj:
            messagebox.showerror("No text column", "Could not find a column of review text."); return
        textcol = obj[0]
    texts = df[textcol].astype(str).tolist()
    probs = classify_many(texts)
    out = pd.DataFrame({"review": texts, "fake_probability": np.round(probs, 3),
                        "verdict": ["LIKELY FAKE" if p >= 0.5 else "LIKELY GENUINE" for p in probs]})
    batch_state["df"] = out
    for r in tv.get_children(): tv.delete(r)
    for n, row in out.iterrows():
        short = row["review"][:70] + ("..." if len(row["review"]) > 70 else "")
        tv.insert("", "end", values=(short, row["verdict"], f"{row['fake_probability']:.0%}"),
                  tags=("fake" if row["verdict"] == "LIKELY FAKE" else "genuine",))
    nfake = int((out["verdict"] == "LIKELY FAKE").sum())
    batch_info.set(f"{len(out)} reviews scanned   \u2014   {nfake} flagged as likely fake,  {len(out) - nfake} genuine")

def export_batch():
    if batch_state["df"] is None:
        messagebox.showinfo("Nothing to export", "Load and scan a CSV first."); return
    path = filedialog.asksaveasfilename(defaultextension=".csv", initialfile="batch_results.csv", filetypes=[("CSV files", "*.csv")])
    if path:
        batch_state["df"].to_csv(path, index=False)
        messagebox.showinfo("Saved", "Results saved to:\n" + path)

tk.Button(bbar, text="Load CSV", font=("Segoe UI", 10, "bold"), fg="white", bg=ACCENT, relief="flat",
          padx=16, pady=7, cursor="hand2", command=load_batch).pack(side="left")
tk.Button(bbar, text="Export results", font=("Segoe UI", 10), fg=TEXT, bg="#e2e8f0", relief="flat",
          padx=16, pady=7, cursor="hand2", command=export_batch).pack(side="left", padx=8)
tk.Label(p2, textvariable=batch_info, font=("Segoe UI", 10), fg=MUTED, bg=BG).pack(anchor="w", padx=28, pady=(10, 6))
btc = card(p2); btc.pack(fill="both", expand=True, padx=28, pady=(0, 22))
tv = ttk.Treeview(btc, columns=("review", "verdict", "prob"), show="headings", style="Modern.Treeview")
tv.heading("review", text="Review"); tv.column("review", width=480, anchor="w")
tv.heading("verdict", text="Verdict"); tv.column("verdict", width=140, anchor="center")
tv.heading("prob", text="Fake %"); tv.column("prob", width=90, anchor="center")
tv.tag_configure("fake", foreground=RED); tv.tag_configure("genuine", foreground=GREEN)
sb = ttk.Scrollbar(btc, orient="vertical", command=tv.yview); tv.configure(yscrollcommand=sb.set)
tv.pack(side="left", fill="both", expand=True, padx=(10, 0), pady=10); sb.pack(side="right", fill="y", pady=10)

p3 = new_page(); pages["dataset"] = p3
page_title(p3, "Dataset", "The labelled Namibian review dataset built for this study.")
try:
    df = pd.read_csv("namibian_reviews_final.csv")
    total = len(df); fl = pd.to_numeric(df["final_label"], errors="coerce")
    genuine = int((fl == 0).sum()); fake = int((fl == 1).sum()); kappa_txt = "n/a"
    if "label_annotator1" in df.columns and "label_annotator2" in df.columns:
        try:
            from sklearn.metrics import cohen_kappa_score
            a1 = pd.to_numeric(df["label_annotator1"], errors="coerce")
            a2 = pd.to_numeric(df["label_annotator2"], errors="coerce")
            m = a1.notna() & a2.notna()
            kappa_txt = f"{cohen_kappa_score(a1[m].astype(int), a2[m].astype(int)):.3f}"
        except Exception: pass
    row = tk.Frame(p3, bg=BG); row.pack(fill="x", padx=22)
    for i, (v, l, col) in enumerate([(str(total), "Total reviews", ACCENT), (str(genuine), "Genuine", GREEN),
                                     (str(fake), "Fake", RED), (kappa_txt, "Cohen's Kappa", "#7c3aed")]):
        stat_card(row, v, l, col).grid(row=0, column=i, padx=6, sticky="ew"); row.grid_columnconfigure(i, weight=1)
    cols = tk.Frame(p3, bg=BG); cols.pack(fill="both", expand=True, padx=22, pady=16)
    cols.grid_columnconfigure(0, weight=1); cols.grid_columnconfigure(1, weight=1)
    if "product_category" in df.columns:
        c1 = card(cols); c1.grid(row=0, column=0, padx=6, sticky="nsew")
        tk.Label(c1, text="By business category", font=("Segoe UI", 11, "bold"), fg=TEXT, bg=CARD).pack(anchor="w", padx=16, pady=(14, 6))
        for cat, n in df["product_category"].value_counts().items():
            tk.Label(c1, text=f"{cat}:  {n}", font=("Segoe UI", 10), fg="#334155", bg=CARD).pack(anchor="w", padx=16, pady=1)
        tk.Label(c1, text="", bg=CARD).pack(pady=4)
    if "source" in df.columns:
        c2 = card(cols); c2.grid(row=0, column=1, padx=6, sticky="nsew")
        tk.Label(c2, text="By source", font=("Segoe UI", 11, "bold"), fg=TEXT, bg=CARD).pack(anchor="w", padx=16, pady=(14, 6))
        for src, n in df["source"].value_counts().items():
            tk.Label(c2, text=f"{src}:  {n}", font=("Segoe UI", 10), fg="#334155", bg=CARD).pack(anchor="w", padx=16, pady=1)
        tk.Label(c2, text="Two annotators labelled independently;\nKappa measures their agreement.",
                 font=("Segoe UI", 9), fg=MUTED, bg=CARD, justify="left").pack(anchor="w", padx=16, pady=(10, 14))
except Exception as e:
    tk.Label(p3, text="Could not load namibian_reviews_final.csv\n" + str(e), font=("Segoe UI", 10), fg=RED, bg=BG, justify="left").pack(anchor="w", padx=28)

p4 = new_page(); pages["results"] = p4
page_title(p4, "Model Results", "Five-fold cross-validated performance on the fake class.")
try:
    dfr = pd.read_csv("results_model_comparison.csv")
    tc = card(p4); tc.pack(fill="x", padx=28, pady=(0, 10))
    cc = list(dfr.columns)
    tv2 = ttk.Treeview(tc, columns=cc, show="headings", height=len(dfr), style="Modern.Treeview")
    for c in cc:
        tv2.heading(c, text=c.replace("_", " ").title()); tv2.column(c, width=150, anchor="center")
    for n, (_, r) in enumerate(dfr.iterrows()):
        tv2.insert("", "end", values=[r[c] for c in cc], tags=("odd" if n % 2 else "even",))
    tv2.tag_configure("odd", background="#f8fafc"); tv2.tag_configure("even", background=CARD)
    tv2.pack(fill="x", padx=10, pady=10)
    cal = tk.Frame(p4, bg="#eff6ff", highlightbackground="#bfdbfe", highlightthickness=1); cal.pack(fill="x", padx=28, pady=6)
    tk.Label(cal, text="Best model:  Logistic Regression  \u2014  highest F1 on the fake class",
             font=("Segoe UI", 10, "bold"), fg="#1e40af", bg="#eff6ff").pack(anchor="w", padx=14, pady=10)
except Exception as e:
    tk.Label(p4, text="Could not load results_model_comparison.csv\n" + str(e), font=("Segoe UI", 10), fg=RED, bg=BG, justify="left").pack(anchor="w", padx=28)

p5 = new_page(); pages["features"] = p5
page_title(p5, "Feature Importance", "What the model learned  \u2014  red points to fake, green points to genuine.")
try:
    dff = pd.read_csv("results_feature_importance.csv")
    fc = card(p5); fc.pack(fill="both", expand=True, padx=28, pady=(0, 22))
    cv = tk.Canvas(fc, bg=CARD, highlightthickness=0); cv.pack(fill="both", expand=True, padx=14, pady=14)
    coefs = pd.to_numeric(dff["coefficient"], errors="coerce").fillna(0)
    maxabs = max(abs(coefs.min()), abs(coefs.max()), 0.001); base = 190; y = 24
    for _, r in dff.iterrows():
        name = str(r["feature"]); c = float(r["coefficient"]); length = int(abs(c) / maxabs * 230)
        colour = "#dc2626" if c >= 0 else "#16a34a"; tag = "FAKE" if c >= 0 else "GENUINE"
        cv.create_text(24, y + 11, text=name, anchor="w", font=("Segoe UI", 11))
        cv.create_rectangle(base, y, base + length, y + 22, fill=colour, outline="")
        cv.create_text(base + length + 10, y + 11, text=f"{c:+.2f}  {tag}", anchor="w", font=("Consolas", 9), fill="#334155")
        y += 42
except Exception as e:
    tk.Label(p5, text="Could not load results_feature_importance.csv\n" + str(e), font=("Segoe UI", 10), fg=RED, bg=BG, justify="left").pack(anchor="w", padx=28)

show("detector")
root.mainloop()
'''

with open("gui.py", "w", encoding="utf-8") as f:
    f.write(gui_code)
print("gui.py upgraded: Evidence + Batch Scan added.")

gui.py upgraded: Evidence + Batch Scan added.


In [2]:
import sys
!"{sys.executable}" gui.py

In [ ]:
import sys
!"{sys.executable}" gui.py

In [1]:
print("hello")

hello


In [2]:
# ============ EDIT: your full slang list ============
SLANG = [
    "lekker","sharp sharp","sharp","the plug","no cap","hey","proper",
    "bru","yoh","eish","giving","clean clean","boss","nxa","mooi","best best","shuu","shomafa","SHIWA NEE",
    # --- add more Namibian slang below, each "in quotes", comma after ---

]
# ====================================================
import re, numpy as np, pandas as pd, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.sparse import hstack, csr_matrix
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v=SentimentIntensityAnalyzer()
    def sentiment(t): return _v.polarity_scores(t)["compound"]
except Exception:
    def sentiment(t): return 0.0

FEATNAMES=["word_count","char_count","exclamations","caps_ratio","slang_count","sentiment"]
df=pd.read_csv("namibian_reviews_final.csv"); df["text"]=df["text"].astype(str)
y=df["final_label"].astype(int).values; raw=df["text"].values

def clean_text(t):
    t=t.lower(); t=re.sub(r"http\S+"," ",t)
    t=re.sub(r"[^a-z0-9\s!]"," ",t); return re.sub(r"\s+"," ",t).strip()
def eng(texts):
    out=[]
    for t in texts:
        low,cc=t.lower(),len(t)
        out.append([len(t.split()),cc,t.count("!"),
            (sum(1 for c in t if c.isupper())/cc if cc else 0),
            sum(low.count(s) for s in SLANG), sentiment(t)])
    return np.array(out,dtype=float)
clean=np.array([clean_text(t) for t in raw])

models={
 "Logistic Regression": lambda: LogisticRegression(max_iter=1000,class_weight="balanced"),
 "Random Forest": lambda: RandomForestClassifier(n_estimators=200,random_state=42,class_weight="balanced"),
 "Naive Bayes": lambda: MultinomialNB(),
}
def run_cv(use):
    skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    res={m:{"acc":[],"prec":[],"rec":[],"f1":[]} for m in models}
    for tr,te in skf.split(clean,y):
        vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=2000)
        Xtr,Xte=vec.fit_transform(clean[tr]),vec.transform(clean[te])
        if use:
            sc=MinMaxScaler()
            Xtr=hstack([Xtr,csr_matrix(sc.fit_transform(eng(raw[tr])))])
            Xte=hstack([Xte,csr_matrix(sc.transform(eng(raw[te])))])
        for name,make in models.items():
            clf=make();clf.fit(Xtr,y[tr]);p=clf.predict(Xte)
            res[name]["acc"].append(accuracy_score(y[te],p))
            res[name]["prec"].append(precision_score(y[te],p,zero_division=0))
            res[name]["rec"].append(recall_score(y[te],p,zero_division=0))
            res[name]["f1"].append(f1_score(y[te],p,zero_division=0))
    return res
rows=[]
for flag,label in [(False,"TF-IDF only"),(True,"TF-IDF + features")]:
    r=run_cv(flag)
    for name,m in r.items():
        rows.append({"feature_set":label,"model":name,
         "accuracy":round(np.mean(m["acc"]),3),"precision":round(np.mean(m["prec"]),3),
         "recall":round(np.mean(m["rec"]),3),"f1":round(np.mean(m["f1"]),3)})
pd.DataFrame(rows).to_csv("results_model_comparison.csv",index=False)

vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=2000)
Xall=vec.fit_transform(clean)
sc=MinMaxScaler(); Xall_f=hstack([Xall,csr_matrix(sc.fit_transform(eng(raw)))])
lr=LogisticRegression(max_iter=1000,class_weight="balanced").fit(Xall_f,y)
pd.DataFrame({"feature":FEATNAMES,"coefficient":np.round(lr.coef_[0][-6:],3)}
            ).sort_values("coefficient",ascending=False).to_csv("results_feature_importance.csv",index=False)
joblib.dump(lr,"demo_model.pkl"); joblib.dump(vec,"demo_vectorizer.pkl"); joblib.dump(sc,"demo_scaler.pkl")

try:
    s=open("gui.py","r",encoding="utf-8").read()
    s=re.sub(r"SLANG = \[[^\]]*\]","SLANG = "+repr(SLANG),s,count=1)
    open("gui.py","w",encoding="utf-8").write(s); synced="gui.py synced"
except Exception as e:
    synced="gui.py not synced: "+str(e)

print("Slang words:",len(SLANG))
print(pd.DataFrame(rows).to_string(index=False))
print(synced, "| model + results updated")

Slang words: 19
      feature_set               model  accuracy  precision  recall    f1
      TF-IDF only Logistic Regression     0.739      0.583   0.500 0.532
      TF-IDF only       Random Forest     0.704      0.500   0.246 0.320
      TF-IDF only         Naive Bayes     0.678      0.300   0.057 0.094
TF-IDF + features Logistic Regression     0.704      0.529   0.500 0.513
TF-IDF + features       Random Forest     0.678      0.287   0.143 0.184
TF-IDF + features         Naive Bayes     0.678      0.200   0.029 0.050
gui.py synced | model + results updated


In [1]:
%pip install vaderSentiment

  Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl.metadata (572 bytes)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-win_amd64.whl.metadata (47 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.2-cp314-cp314-win_amd64.whl (210 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)

   ---------------------------------------- 0/6 [urllib3]
   ------------- -------------------------- 2/6 [charset_normalizer]
   --------------------------------- ------ 5/6 [vaderSentiment]
   ---------------------------------------- 6/6 [vaderSentiment]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
!"{sys.executable}" regenerate_model.py

Traceback (most recent call last):
  File "c:\Users\Acer\Documents\school\Research Project\Fake Review Detector\regenerate_model.py", line 14, in <module>
    import matplotlib; matplotlib.use("Agg")
    ^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'matplotlib'


In [3]:
import sys
!"{sys.executable}" regenerate_model.py

VADER sentiment: WORKING
      feature_set               model  accuracy  precision  recall    f1
      TF-IDF only Logistic Regression     0.739      0.583   0.500 0.532
      TF-IDF only          Linear SVM     0.704      0.525   0.471 0.491
      TF-IDF only       Random Forest     0.704      0.500   0.246 0.320
      TF-IDF only         Naive Bayes     0.678      0.300   0.057 0.094
TF-IDF + features Logistic Regression     0.809      0.650   0.829 0.727
TF-IDF + features          Linear SVM     0.730      0.565   0.607 0.580
TF-IDF + features       Random Forest     0.687      0.420   0.225 0.285
TF-IDF + features         Naive Bayes     0.678      0.200   0.029 0.050
     feature  coefficient
  word_count       -0.533
  char_count       -0.534
exclamations        0.499
  caps_ratio        0.073
 slang_count        0.403
   sentiment        2.491


Traceback (most recent call last):
  File "c:\Users\Acer\Documents\school\Research Project\Fake Review Detector\regenerate_model.py", line 99, in <module>
    fig,ax=plt.subplots(figsize=(4.4,4)); ax.imshow(cm,cmap="Blues"); ax.set_xticks([0,1]); ax.set_yticks([0,1])
           ^^^
NameError: name 'plt' is not defined


In [4]:
import sys
!"{sys.executable}" gui_v3.py

In [5]:
import sys
!"{sys.executable}" gui_v3.py

In [2]:
%cd "C:\Users\Acer\Documents\school\Research Project\Fake Review Detector\extension_project\api"

C:\Users\Acer\Documents\school\Research Project\Fake Review Detector\extension_project\api
